# **Initialization**

In [4]:
print("Start")

Start


In [5]:
import numpy as np
import math
import random
import sys
import os
import modified_didppy as m_dp# Assuming you have your custom DIDPPy installed

# **0. DIDP model declaration and dual bound expressions function**

In [6]:
def example_creation_of_didp_model_function():
    # [Data definition]
    #...
    #[DIDP Model Creation]
    #...
    # [Return variables needed for dual bounds calculation]
    # didp_model, = model, 
    # didp_model_metadata_dict = {"name_of_variable/constants": actual_name_in_modeling}
    return None # didp_model, didp_model_metadata_dict

def example_of_dual_bound_expression_function(didp_model, didp_model_metadata_dict):
    # [Variables/data definition]
    #...
    #[Dual bound logic declaration]
    #...
    # [Return a dual bound registry dict]
    # dual_bound_dict = automatic_creation_of_dual_bounds_registry(locals())
    return None # dual_bound_dict

# **1. Global Helper functions**

In [7]:
# ==========================================
# 1. EVOLUTIONARY ALGORITHM HYPERPARAMETERS
# ==========================================
POPULATION_SIZE = 50        # Size of the population in each generation
GENERATIONS = 100           # Number of generations to run
MUTATION_RATE = 0.2         # Probability of mutating an individual
CROSSOVER_RATE = 0.8        # Probability of performing crossover
ELITISM_RATE = 0.01
# ==========================================
# 2. CHROMOSOME GENERATION CONSTRAINTS
# ==========================================
# Bounds for the coefficients generated for weighted blocks (e.g., 5.5 * h1)
LB_range_of_constant = 0.0  
UB_range_of_constant = 10.0 

# Depth limits for the RPN trees (used in Ramped Half-and-Half generator)
min_chromosome_length = 2               # Minimum depth of the initial trees
max_chromosome_length = 6               # Maximum depth of the initial trees
# Mutation: Maximum depth allowed for the *newly generated* subtree during mutation
mutation_max_subtree_depth = random.randint(min_chromosome_length, max_chromosome_length + 4)  # Randomly chosen between 1 and 3

# ==========================================
# 3. PROBLEM DOMAIN (INPUTS)
# ==========================================
# The atomic operations allowed in the RPN expression
available_operations = ["ADD", "SUBTRACT", "MAX", "MIN", "MULTIPLY", "PDIV"]

didp_model_registry=None          # MUST BE PROVIDED: Factory function creation_of_didp_model_function
dual_bound_expression_function=None # MUST BE PROVIDED: Factory function dual_bound_expression_function to produce the dual_bound_functions_registry dictionary
# The dictionary of actual heuristic functions available (h1, h2, etc.)
# You must define the functions h1(state), h2(state) first.
# Example:
# dual_bound_functions_registry = {
#     "h1": h1_function_object,
#     "h2": h2_function_object
# }
dual_bound_functions_registry = {} 

# The Ground Truth optimal cost for the specific problem instance
# Used to calculate fitness (deviation from optimal)
OPTIMAL_COST_REFERENCE = 5 

# Time limit (in seconds) for the DIDP solver to run per chromosome evaluation
SOLVER_TIME_LIMIT = 1 #seconds

# ==========================================
# 4. OPERATOR SPECIFIC PARAMETERS
# ==========================================
# 1-Point Crossover: Probability of using Homology (matching structure) vs Random fallback
homology_1_point_crossover_probability = 0.5

# Subtree Crossover: Probability of swapping a Function (Branch) vs Terminal (Leaf)
subtree_crossover_probability = 0.9

# Uniform Crossover: Probability of swapping genes at a specific index
uniform_crossover_probability = 0.5

In [8]:
#Utility functions
def extract_chromosome_from_chromosome_fitness_dict(chromosome_fitness_dict):
    """
    Extracts the chromosome from a chromosome-fitness dictionary.
    """
    return chromosome_fitness_dict.get('chromosome')

def automatic_creation_of_dual_bounds_registry(local_scope):
    """
    Automatically builds a registry dict from the local scope.
    It selects all variables that:
    1. Start with 'h' (e.g., 'h1', 'h_unvisited')
    2. Are callable (are functions)
    """
    return {
        name: func 
        for name, func in local_scope.items() 
        if name.startswith("h") and callable(func)
    }

def is_operator(gene, available_operations):
    return gene in available_operations

def analyzing_chromosome_based_on_rpn_structure(chromosome_fitness_dict, available_operations = available_operations):
    """
    Scans a chromosome to identify all valid subtrees.
    Returns a dictionary categorizing them into 'TERMINALS' and 'FUNCTIONS'.
    """
    
    chromosome = extract_chromosome_from_chromosome_fitness_dict(chromosome_fitness_dict)
    # Stores tuples of (start_index, end_index)
    structure = {
        "TERMINALS": [],  # Leaves or Weighted Blocks [c, h, *]
        "FUNCTIONS": []   # Complex operators (ADD, MAX, etc)
    }
    
    # We scan backwards (Right to Left) because RPN roots are on the right
    i = len(chromosome) - 1
    
    while i >= 0:
        root_gene = chromosome[i]
        end_index = i
        
        # Logic to find the start_index of the subtree
        required_inputs = 1 # The root needs to be produced
        current_pos = i
        
        while required_inputs > 0:
            gene = chromosome[current_pos]
            # If the sub tree is an Operator, it needs 2 input, so we produce 1 more requirement (Net +1 dependency)
            if is_operator(gene, available_operations):
                required_inputs += 1 
            else:
                required_inputs -= 1 # Terminal satisfies only 1 dependent component
            current_pos -= 1
        start_index = current_pos + 1
        
        # --- CATEGORIZATION LOGIC ---
        subtree_slice = chromosome[start_index : end_index+1]
        
        # Check if it is a "Weighted Terminal Block" [float, str, MULTIPLY]
        # The prompt specifically asks to treat these as terminals
        is_weighted_block = (
            len(subtree_slice) == 3 and 
            isinstance(subtree_slice[0], (int, float)) and
            isinstance(subtree_slice[1], str) and 
            subtree_slice[2] == "MULTIPLY"
        )
        
        # Check if it is a raw terminal (single item)
        is_raw_terminal = (start_index == end_index)
        
        if is_weighted_block or is_raw_terminal:
            structure["TERMINALS"].append((start_index, end_index))
        else:
            structure["FUNCTIONS"].append((start_index, end_index))
            
        # Move to the next node to the left
        i -= 1
        
        # Optimization: The loop above `while i >= 0` scans every index.
        # The inner logic finds the subtree rooted at `i`.
        # Since every index in a valid RPN is the root of *some* subtree (even if just itself),
        # we let the loop continue naturally.

    return structure

# **2. Decoding chromosome to create useable function**

In [9]:
"""Example Trace for ['h1', 'h3', 'MAX', 2, 'ADD'] with h1=10, h3=5:
  *'h1': stack.append(10) -> stack is [10]
  *'h3': stack.append(5) -> stack is [10, 5]
  *'MAX': b = stack.pop() (5), a = stack.pop() (10). max(10, 5) is 10. stack.append(10) -> stack is [10]
  *2: stack.append(2) -> stack is [10, 2]
  *'ADD': b = stack.pop() (2), a = stack.pop() (10). 10 + 2 is 12. stack.append(12) -> stack is [12]"""

"Example Trace for ['h1', 'h3', 'MAX', 2, 'ADD'] with h1=10, h3=5:\n  *'h1': stack.append(10) -> stack is [10]\n  *'h3': stack.append(5) -> stack is [10, 5]\n  *'MAX': b = stack.pop() (5), a = stack.pop() (10). max(10, 5) is 10. stack.append(10) -> stack is [10]\n  *2: stack.append(2) -> stack is [10, 2]\n  *'ADD': b = stack.pop() (2), a = stack.pop() (10). 10 + 2 is 12. stack.append(12) -> stack is [12]"

## Converting the chromosome to useable function

In [10]:
def convert_chromosome_to_string_of_python_code(chromosome):
    """
    Translates a list-based chromosome (RPN) into a Python function definition string.
    """
    stack = []
    try:
        for gene in chromosome:
            if isinstance(gene, (int, float)):
                stack.append(str(gene))
            elif isinstance(gene, str):
                if gene == "ADD":
                    b, a = stack.pop(), stack.pop()
                    stack.append(f"({a} + {b})")
                elif gene == "SUBTRACT":
                    b, a = stack.pop(), stack.pop()
                    stack.append(f"({a} - {b})")
                elif gene == "MULTIPLY":
                    b, a = stack.pop(), stack.pop()
                    stack.append(f"({a} * {b})")
                elif gene == "MAX":
                    b, a = stack.pop(), stack.pop()
                    stack.append(f"max({a}, {b})")
                elif gene == "MIN":
                    b, a = stack.pop(), stack.pop()
                    stack.append(f"min({a}, {b})")
                elif gene == "PDIV":
                    b, a = stack.pop(), stack.pop()
                    # Protected Division Logic:
                    # If denominator (b) is close to 0, return 1.0 (identity), otherwise divide.
                    # We use an inline ternary operator for the string generation.
                    stack.append(f"({a} / {b} if abs({b}) > 1e-6 else 1.0)")
                else:
                    # It's a heuristic name like "h1"
                    stack.append(f"{gene}(state)")
        if len(stack) == 1:
            formula_body = stack.pop()
            # Define the standard name for our function
            func_name = "dual_bound_combination"
            code_string = f"def {func_name}(state):\n    return {formula_body}"
            return code_string, func_name
        else:
            return None, None
    except IndexError:
        return None, None

def compile_chromosome_to_useable_function(chromosome_fitness_dict, dual_bound_functions_dict, print_code):
    """
    Converts a chromosome list into a real, callable Python function.
    Args:
        chromosome: The list (RPN)
        dual_bound_functions_dict: Dict of actual functions available to the code
        (e.g., {'h1': h1, 'h2': h2})
    """
    chromosome = extract_chromosome_from_chromosome_fitness_dict(chromosome_fitness_dict)
    # 1. Generate the code string
    code_string, func_name = convert_chromosome_to_string_of_python_code(chromosome)
    if code_string is None:
        raise ValueError("Invalid Chromosome")
    if print_code:
      print(f"Generated Code:\n{code_string}\n")

    # 2. Prepare the execution namespace
    # We create a dictionary that holds all the functions our new code needs.
    # This acts like the 'globals()' for the new function.
    execution_namespace = dual_bound_functions_dict.copy()

    # 3. Compile and Execute the string
    # This runs the 'def dual_bound_combination...' string, creating the function
    # inside the execution_namespace dict.
    exec(code_string, execution_namespace)

    # 4. Retrieve the live function object
    callable_function = execution_namespace[func_name]

    return callable_function

##### *Testing the dual bound function achieved from the chromosome*



In [11]:
def h1(state):
    return sum(state) # Example logic
def h2(state):
    return 20
def h3(state):
    return 5
# This is the "input" you wanted to control
dual_bound_functions_registry = {
    "h1": h1,
    "h2": h2,
    "h3": h3
    # You can add "h4", "h5" here anytime
}
# Create a dummy state
state = [1, 2, 3]
chromosome_fitness_dict = {'chromosome': ["h1", 3, "h3", "MULTIPLY", "MAX", 2, "ADD"], 'fitness': []}
print(f"Testing with chromosome: {chromosome_fitness_dict}")
print(f"Initial state values: ")
print(f"  h1(state) = {h1(state)}")
print(f"  h2(state) = {h2(state)}")
print(f"  h3(state) = {h3(state)}")
combined_dual_bound_function = compile_chromosome_to_useable_function(chromosome_fitness_dict, dual_bound_functions_registry, False)
result_1= combined_dual_bound_function(state)
print(result_1)

Testing with chromosome: {'chromosome': ['h1', 3, 'h3', 'MULTIPLY', 'MAX', 2, 'ADD'], 'fitness': []}
Initial state values: 
  h1(state) = 6
  h2(state) = 20
  h3(state) = 5
17


# **3. Encoding function of chromosome**

## Helper functions

In [ ]:
#Generation of termninal block
def generate_random_terminal_block(dual_bound_functions_dict, LB_range_of_constant, UB_range_of_constant):
    """
    Creates a single terminal unit (leaf) for the RPN list.
    It randomly decides whether to wrap the heuristic with a coefficient.
    """
    keys = list(dual_bound_functions_dict.keys())
    h_name = random.choice(keys)
    if random.random() < 0.5:
        # Weighted: [coef, h, "MULTIPLY"]
        coef = round(random.uniform(LB_range_of_constant, UB_range_of_constant), 2)
        return [coef, h_name, "MULTIPLY"]
    else:
        # Raw: [h]
        return [h_name]

def generate_rpn_tree_recursive(current_depth, max_node_depth, method, dual_bound_functions_dict,
                                LB_range_of_constant, UB_range_of_constant, available_operations):
    """
    Recursively builds an RPN list using standard GP growth logic.
    """
    # --- BASE CASE: Hit Depth Limit ---
    if current_depth >= max_node_depth:
        # Must return a terminal
        return generate_random_terminal_block(dual_bound_functions_dict, LB_range_of_constant,
                                            UB_range_of_constant)

    # --- SELECTION: Choose between Function or Terminal ---
    if method == "FULL":
        # FULL: Always branch until max_depth is hit
        choice = "FUNCTION"
    else:
        # GROW: Randomly pick Function or Terminal
        # (Standard GP often uses a probability here, e.g., based on set sizes)
        # Here we use 50/50 for simplicity, or you can weight it.
        choice = random.choice(["FUNCTION", "TERMINAL"])

    # --- CONSTRUCTION ---
    if choice == "TERMINAL":
        return generate_random_terminal_block(dual_bound_functions_dict, LB_range_of_constant,
                                            UB_range_of_constant)

    else: # FUNCTION (Internal Node)
        op = random.choice(available_operations)

        # Recursively generate left and right branches
        # Note: Binary operators always need 2 children
        left_rpn = generate_rpn_tree_recursive(
            current_depth + 1, max_node_depth, method, dual_bound_functions_dict, LB_range_of_constant,
            UB_range_of_constant, available_operations)
        right_rpn = generate_rpn_tree_recursive(
            current_depth + 1, max_node_depth, method, dual_bound_functions_dict, LB_range_of_constant,
            UB_range_of_constant, available_operations)

        # Combine in RPN order: Left, Right, Operator
        return left_rpn + right_rpn + [op]

## General generator

In [ ]:
# ==========================================
# THE GENERAL GENERATOR
# ==========================================
def generate_valid_chromosome(dual_bound_functions_dict, LB_range_of_constant, UB_range_of_constant, 
                              available_operations = available_operations):
    """
    Generates a valid random chromosome using keys from the registry.
    Now supports multiplying bounds together!
    """
    # Extract the keys (for instance ["h1", "h2", "h3"]) dynamically from the dictionary
    available_dual_bound_functions = list(dual_bound_functions_dict.keys())

    # 1. Create initial blocks (Terminals) by iteration through each heuristic to add in coefficients
    blocks = []
    for h_name in available_dual_bound_functions:
        if random.random() < 0.5:
            # Weighted block: [coef, h, "MULTIPLY"] -> resulting in (coef * h)
            coef = round(random.uniform(LB_range_of_constant, UB_range_of_constant), 2)
            blocks.append([coef, h_name, "MULTIPLY"])
        else:
            # Raw block: [h]
            blocks.append([h_name])

    # 2. Merge blocks until one remains
    #Create a copy for the blocks of each heuristic function
    current_pool = blocks.copy()

    while len(current_pool) > 1:
        # Pick two random blocks from the pool
        # Ensure there are at least 2 blocks to sample from
        if len(current_pool) < 2:
            break

        idx1, idx2 = random.sample(range(len(current_pool)), 2)

        # Pop in descending order to handle index shifts
        right = current_pool.pop(max(idx1, idx2))
        left = current_pool.pop(min(idx1, idx2))

        # Merge with random operator
        op = random.choice(available_operations)

        # Structure: [Left Block] + [Right Block] + [Operator]
        merged = left + right + [op]

        current_pool.append(merged)

    return current_pool[0]

##### *Testing general encoding function*

In [ ]:
def h1(state): return sum(state)
def h2(state): return 20
def h3(state): return 5

dual_bound_functions_registry = {
    "h1": h1,
    "h2": h2,
    "h3": h3
}

LB_range_of_constant, UB_range_of_constant = float(0.0), float(10.0)
print("\n--- Test 1: Basic Generation & Structure Check ---")
chrom1 = generate_valid_chromosome(dual_bound_functions_registry, LB_range_of_constant, UB_range_of_constant, available_operations)
print(f"Generated Chromosome: {chrom1}")
chromo_fitness_dict1 = {'chromosome': chrom1, 'fitness': 0}
func1 = compile_chromosome_to_useable_function(chromo_fitness_dict1, dual_bound_functions_registry, print_code = True)
# Check basic properties
assert isinstance(chrom1, list), "Chromosome must be a list"
assert len(chrom1) > 0, "Chromosome must not be empty"
# A valid RPN expression must end with an operator (unless it's a single term, unlikely here)
# or a function if it was a simple weight
last_gene = chrom1[-1]
print(f"Last gene is operator/function: {last_gene in ['ADD', 'SUBTRACT', 'MAX', 'MIN', 'MULTIPLY']}")


print("\n--- Test 2: Stress Test (Generate 100 chromosomes) ---")
valid_count = 0
total_runs = 100

for i in range(total_runs):
    chrom2 = generate_valid_chromosome(dual_bound_functions_registry, LB_range_of_constant, UB_range_of_constant, available_operations)
    chromo_fitness_dict2 = {'chromosome': chrom2, 'fitness': 0}
    func2 = compile_chromosome_to_useable_function(chromo_fitness_dict2, dual_bound_functions_registry, print_code = False)
    res = func2(state)

    if res != float('-inf'):
        valid_count += 1
    else:
        print(f"Failed Chromosome: {chrom2}")

print(f"Validity Rate: {valid_count}/{total_runs}")
assert valid_count == total_runs, "Generator produced invalid RPN structures!"


print("\n--- Test 3: Diversity Check ---")
# Check if different runs produce different chromosomes
chrom3a = generate_valid_chromosome(dual_bound_functions_registry, LB_range_of_constant, UB_range_of_constant, available_operations)
chrom3b = generate_valid_chromosome(dual_bound_functions_registry, LB_range_of_constant, UB_range_of_constant, available_operations)
print(f"Chrom A: {chrom3a}")
chromo_fitness_dict3a = {'chromosome': chrom3a, 'fitness': 0}
func3a = compile_chromosome_to_useable_function(chromo_fitness_dict3a, dual_bound_functions_registry, print_code = True)
print(f"Chrom B: {chrom3b}")
chromo_fitness_dict3b = {'chromosome': chrom3b, 'fitness': 0}
func3a = compile_chromosome_to_useable_function(chromo_fitness_dict3b, dual_bound_functions_registry, print_code = True)
if chrom3a != chrom3b:
    print("PASSED: Generator produces diverse output.")
else:
    print("WARNING: Generator produced identical output (might happen by chance, but unlikely for complex trees)")


print("\n--- Test 4: 'Multiply' Operator Check ---")
# Verify that 'MULTIPLY' appears in the output eventually
# (It should, either as a coefficient weight or a merge operator)
found_multiply = False
for _ in range(20):
    chrom4 = generate_valid_chromosome(dual_bound_functions_registry, LB_range_of_constant, UB_range_of_constant, available_operations)
    if "MULTIPLY" in chrom4:
        found_multiply = True
        break

if found_multiply:
    print("PASSED: 'MULTIPLY' operator found in chromosomes.")
else:
    print("WARNING: 'MULTIPLY' operator not found in 20 runs (check generation logic).")

print("\n--- Test 5: Mathematical Validity (Execution Check) ---")
chrom5=[0.87, 'h1', 'MULTIPLY', 'h2', 6.05, 'h3', 'MULTIPLY', 'MIN', 'MULTIPLY']
chromo_fitness_dict5 = {'chromosome': chrom5, 'fitness': 0}
# Create a dummy state
state = [1, 2, 3, 5]
# Compile the generated chromosome
func5 = compile_chromosome_to_useable_function(chromo_fitness_dict5, dual_bound_functions_registry, print_code = True)
#Using the generated function to calculat the result
result5 = func1(state)
print(f"Executed Result: {result5}")

print("\n--- All Tests Completed ---")


--- Test 1: Basic Generation & Structure Check ---
Generated Chromosome: ['h1', 'h2', 8.21, 'h3', 'MULTIPLY', 'MAX', 'MAX']
Generated Code:
def dual_bound_combination(state):
    return max(h1(state), max(h2(state), (8.21 * h3(state))))

Last gene is operator/function: True

--- Test 2: Stress Test (Generate 100 chromosomes) ---
Validity Rate: 100/100

--- Test 3: Diversity Check ---
Chrom A: ['h2', 9.81, 'h1', 'MULTIPLY', 6.34, 'h3', 'MULTIPLY', 'PDIV', 'MIN']
Generated Code:
def dual_bound_combination(state):
    return min(h2(state), ((9.81 * h1(state)) / (6.34 * h3(state)) if abs((6.34 * h3(state))) > 1e-6 else 1.0))

Chrom B: ['h1', 'h2', 'h3', 'MULTIPLY', 'PDIV']
Generated Code:
def dual_bound_combination(state):
    return (h1(state) / (h2(state) * h3(state)) if abs((h2(state) * h3(state))) > 1e-6 else 1.0)

PASSED: Generator produces diverse output.

--- Test 4: 'Multiply' Operator Check ---
PASSED: 'MULTIPLY' operator found in chromosomes.

--- Test 5: Mathematical Validity (

## Ramped-Half-and-Half Generator

In [ ]:
# ==========================================
# MAIN GENERATOR
# ==========================================
def generate_ramped_half_and_half(dual_bound_functions_dict, LB_range_of_constant, UB_range_of_constant,
                                  min_chromosome_length, max_chromosome_length, available_operations = available_operations):
    """
    Generates a chromosome (RPN list) using the Ramped Half-and-Half method.

    Args:
        min_depth = min_chromosome_length (int): Minimum tree depth.
        max_depth = max_chromosome_length (int): Maximum tree depth.
    """

    # 1. RAMPED: Pick a random max depth for this individual
    # This ensures the population has a mix of shallow and deep trees.
    target_depth = random.randint(min_chromosome_length, max_chromosome_length)

    # 2. HALF-AND-HALF: Choose method
    # 50% chance for "GROW", 50% chance for "FULL"
    method = "GROW" if random.random() < 0.5 else "FULL"

    # 3. Generate
    # We start at depth 0
    chromosome = generate_rpn_tree_recursive(
        0,
        target_depth,
        method,
        dual_bound_functions_dict,
        LB_range_of_constant,
        UB_range_of_constant,
        available_operations
    )
    return chromosome

##### *Testing Ramped Half-and-half encoding function*

In [ ]:
def h1(state): return sum(state)
def h2(state): return 20
def h3(state): return 5

dual_bound_functions_registry = {
    "h1": h1,
    "h2": h2,
    "h3": h3
}

# Configuration
LB_range_of_constant = 0.0
UB_range_of_constant = 10.0
min_chromosome_length= 2
max_chromosome_length = 6
available_operations = ["ADD", "SUBTRACT", "MAX", "MIN", "MULTIPLY"]

print("\n=== Test 1: Basic Generation ===")
chrom = generate_ramped_half_and_half(
    dual_bound_functions_registry, LB_range_of_constant, UB_range_of_constant,
    min_chromosome_length, max_chromosome_length, available_operations)
print(f"Chromosome: {chrom}")
assert isinstance(chrom, list)
assert len(chrom) > 0


print("\n=== Test 2: Validity Stress Test (100 Runs) ===")
# This checks if the recursive logic ALWAYS produces valid RPN (stack balanced)
errors = 0
for i in range(100):
    chrom = generate_ramped_half_and_half(
        dual_bound_functions_registry, LB_range_of_constant, UB_range_of_constant,
        min_chromosome_length, max_chromosome_length, available_operations
    )
    chromosome_fitness_dict = {'chromosome': chrom, 'fitness': 0}
    try:
        # If this runs without error, the RPN stack logic is valid
        func = compile_chromosome_to_useable_function(chromosome_fitness_dict, dual_bound_functions_registry,
                                                    print_code = False)
    except ValueError:
        errors += 1
        print(f"Invalid Chromosome Generated: {chrom}")

print(f"Errors found: {errors}/100")
assert errors == 0, "Generator produced invalid chromosomes!"


print("\n=== Test 3: Diversity/Ramping Check ===")
# Check if we get different lengths (implies different depths/structures)
lengths = set()
for i in range(20):
    chrom = generate_ramped_half_and_half(
        dual_bound_functions_registry, LB_range_of_constant, UB_range_of_constant,
        min_chromosome_length, max_chromosome_length, available_operations
    )
    lengths.add(len(chrom))

print(f"Unique chromosome lengths found in 20 runs: {lengths}")
if len(lengths) > 1:
    print("PASSED: Population shows structural diversity (Ramping/Grow worked).")
else:
    print("WARNING: All chromosomes were the same length (Check random logic).")


print("\n=== Test 4: Execution Check ===")
# Take a generated chromosome and actually run it on a state
state = [1, 2, 3] # sum = 6
chrom = generate_ramped_half_and_half(
    dual_bound_functions_registry, LB_range_of_constant, UB_range_of_constant,
    min_chromosome_length, max_chromosome_length, available_operations
)
chromosome_fitness_dict = {'chromosome': chrom, 'fitness': 0}
func = compile_chromosome_to_useable_function(chromosome_fitness_dict, dual_bound_functions_registry,
                                              print_code = True)
result = func(state)
print(f"State: {state}")
print(f"Heuristics: h1={h1(state)}, h2={h2(state)}, h3={h3(state)}")
print(f"Formula (RPN): {chrom}")
print(f"Result: {result}")
assert isinstance(result, (int, float)), "Result should be a number"

print("\n=== All Tests Completed} ===")


=== Test 1: Basic Generation ===
Chromosome: [6.64, 'h1', 'MULTIPLY', 2.05, 'h1', 'MULTIPLY', 1.41, 'h1', 'MULTIPLY', 'SUBTRACT', 'MULTIPLY', 'h3', 'MAX']

=== Test 2: Validity Stress Test (100 Runs) ===
Errors found: 0/100

=== Test 3: Diversity/Ramping Check ===
Unique chromosome lengths found in 20 runs: {1, 97, 99, 195, 7, 9, 45, 15, 79, 49, 17, 21, 53, 87, 27, 29, 95}
PASSED: Population shows structural diversity (Ramping/Grow worked).

=== Test 4: Execution Check ===
Generated Code:
def dual_bound_combination(state):
    return (min(max(max(max(h3(state), (9.12 * h3(state))), ((8.87 * h2(state)) - h3(state))), min((h3(state) - (5.04 * h2(state))), max((9.7 * h1(state)), h1(state)))), min((max(h2(state), h1(state)) * (h1(state) + h1(state))), min(min(h2(state), (4.26 * h3(state))), ((8.32 * h1(state)) + h2(state))))) + ((max(max((2.97 * h1(state)), h1(state)), ((3.04 * h3(state)) - (5.26 * h1(state)))) * min((h1(state) - h3(state)), max((2.1 * h2(state)), (6.12 * h3(state))))) + 

## Combined generator

In [ ]:
def generate_combined_chromosome(dual_bound_functions_dict, LB_range_of_constant, UB_range_of_constant,
                                min_chromosome_length, max_chromosome_length, available_operations = available_operations):
    """
    Generates a chromosome using a combined strategy:
    - 70% chance to use the General Generator (generate_valid_chromosome)
    - 30% chance to use the Ramped Half-and-Half Generator (generate_ramped_half_and_half)
    """
    if random.random() < 0.7:
        # Use General Generator
        chromosome = generate_valid_chromosome(dual_bound_functions_dict, LB_range_of_constant, UB_range_of_constant, available_operations)
        chromosome_fitness_dict = {'chromosome': chromosome, 'fitness': 0}
        return chromosome_fitness_dict
    else:
        # Use Ramped Half-and-Half Generator
        chromosome = generate_ramped_half_and_half(dual_bound_functions_dict, LB_range_of_constant, UB_range_of_constant, 
                                                   min_chromosome_length, max_chromosome_length, available_operations)
        chromosome_fitness_dict = {'chromosome': chromosome, 'fitness': 0}
        return chromosome_fitness_dict

#### *Testing combined generator*

In [ ]:
chrom_list = []

def h1(state): return sum(state)
def h2(state): return 20
def h3(state): return 5

dual_bound_functions_registry = {
    "h1": h1,
    "h2": h2,
    "h3": h3
}

for i in range(0,10):
  test_chromosome_fitness_dict = generate_combined_chromosome(
      dual_bound_functions_registry,
      LB_range_of_constant,
      UB_range_of_constant,
      min_chromosome_length,
      min_chromosome_length,
      available_operations
  )
  test_chromosome = extract_chromosome_from_chromosome_fitness_dict(test_chromosome_fitness_dict)
  chrom_list.append(test_chromosome_fitness_dict)
  print(f"Generated Chromosome: {test_chromosome}")
  compiled_func = compile_chromosome_to_useable_function(test_chromosome_fitness_dict, dual_bound_functions_registry, print_code=True)

# Also test compilation to ensure validity
try:
  for test_chromosome_fitness_dict in chrom_list:
    compiled_func = compile_chromosome_to_useable_function(test_chromosome_fitness_dict, dual_bound_functions_registry, print_code=False)
    test_state = [1, 2, 3]
    result = compiled_func(test_state)
    print(f"Compiled function result with state {test_state}: {result}")
except Exception as e:
    print(f"Error compiling or executing chromosome: {e}")

Generated Chromosome: ['h1', 6.59, 'h2', 'MULTIPLY', 'h3', 'ADD', 'MULTIPLY']
Generated Code:
def dual_bound_combination(state):
    return (h1(state) * ((6.59 * h2(state)) + h3(state)))

Generated Chromosome: ['h3', 'h1', 'h2', 'MIN', 'SUBTRACT']
Generated Code:
def dual_bound_combination(state):
    return (h3(state) - min(h1(state), h2(state)))

Generated Chromosome: [1.66, 'h1', 'MULTIPLY', 'h2', 0.98, 'h3', 'MULTIPLY', 'SUBTRACT', 'MAX']
Generated Code:
def dual_bound_combination(state):
    return max((1.66 * h1(state)), (h2(state) - (0.98 * h3(state))))

Generated Chromosome: [6.2, 'h1', 'MULTIPLY', 0.11, 'h2', 'MULTIPLY', 7.33, 'h3', 'MULTIPLY', 'MIN', 'MIN']
Generated Code:
def dual_bound_combination(state):
    return min((6.2 * h1(state)), min((0.11 * h2(state)), (7.33 * h3(state))))

Generated Chromosome: [4.16, 'h1', 'MULTIPLY', 3.23, 'h2', 'MULTIPLY', 9.12, 'h3', 'MULTIPLY', 'MULTIPLY', 'MULTIPLY']
Generated Code:
def dual_bound_combination(state):
    return ((4.16 * h1(

# **4. Calculating fitness of decoded functions**

## Core functions

In [ ]:
def combining_modified_didppy_solver_with_chromosome(chromosome, didp_model_registry, dual_bound_functions_expression, time_limit = SOLVER_TIME_LIMIT):
    """
    Runs the DIDP solver with a flexible, evolved heuristic.
    
    Args:
        chromosome (list): The RPN list (e.g., ['h1', 'h2', 'ADD']).
        model_factory (func): Returns (model, model_vars).
        heuristic_factory (func): Accepts (model, model_vars) and returns 
                                the dual_bound_registry dict.
    """
    
    # --- 1. Create Fresh Model Instance ---
    didp_bundle = didp_model_registry() #This is a function contain the DIDP model creation logic
    didp_model = didp_bundle[0]
    # --- 2. Create Linked Heuristics ---
    # We call the factory to create the base heuristics (h1, h2, etc.)
    # and bind them to *this specific model's* variables.
    dual_bound_registry = dual_bound_functions_expression(didp_bundle)
    
    # --- 3. Compile Combined Bound ---
    # This is the SINGLE dual bound used by the solver
    try:
        # === FIX IS HERE ===
        # Wrap the list in a dict because the compiler expects {'chromosome': ...}
        temp_individual_dict = {'chromosome': chromosome}
        combined_bound_func = compile_chromosome_to_useable_function(
            temp_individual_dict, 
            dual_bound_registry, 
            print_code=False
        )
    except Exception as e:
        print(f"Heuristic Compilation Failed: {e}")
        return float('inf')

    # --- 4. Safety Wrapper ---
    # Catches runtime math errors (e.g., div by zero) inside the search
    def safe_dual_bound(state):
        try:
            val = combined_bound_func(state)
            return float(val)
        except Exception:
            return 0.0

    # --- 5. Run Solver ---
    try:  
        solver = m_dp.CustomDualBoundCABSv1(
            didp_model, 
            dual_bound_func=safe_dual_bound, 
            quiet=True,
            time_limit = time_limit, # seconds
        )
        solution = solver.search()
        
        if solution.cost is not None:
            return solution.cost
        else:
            return float('inf')

    except Exception as e:
        print(f"Solver Error: {e}")
        return float('inf')

def example_creation_of_didp_model_function():
    # [Data definition]
    #...
    #[DIDP Model Creation]
    #...
    # [Return variables needed for dual bounds calculation]
    # didp_model, = model 
    # didp_model_metadata_dict = {"name_of_variable/constants": actual_name_in_modeling}
    return None # didp_model, didp_model_metadata_dict

def example_of_dual_bound_expression_function(didp_model, didp_model_metadata_dict):
    # [Variables/data definition]
    #...
    #[Dual bound logic declaration]
    #...
    # [Return a dual bound registry dict]
    # dual_bound_dict = automatic_creation_of_dual_bounds_registry(locals())
    return None # dual_bound_dict

# Run the Solver
# result = combining_modified_didppy_solver_with_chromosome(
#    chromosome, 
#    create_cvrp_example_creation_of_didp_modeldidp_model,       # Your model logic
#    example_creation_of_didp_model   # Your heuristic logic)

def chromosome_fitness_dict_evaluation(chromosome_fitness_dict, didp_model_registry, dual_bound_expression_function, reference_point):
    """
    Evaluates a chromosome by running the DIDP solver and comparing the result to a reference point.
    Updates the 'fitness' key in the input dictionary.

    Args:
        chromosome_fitness_dict (dict): {'chromosome': [...], 'fitness': None}
        didp_model_registry (func): Function to create DIDP model & meatdata (variables, state variables).
        dual_bound_functions_expression (func): Function to create dual bound registry.
        reference_point (float): The known optimal cost (Ground Truth).
        
    Returns:
        dict: The updated chromosome_fitness_dict with calculated fitness.
    """
    
    # 1. Extract the chromosome list
    chromosome = chromosome_fitness_dict.get('chromosome')
    if not chromosome:
        chromosome_fitness_dict['fitness'] = float('inf')
        return chromosome_fitness_dict

    # 2. Run the Solver Pipeline
    # We use the combining function to get the actual objective value found by the solver
    solver_result_cost = combining_modified_didppy_solver_with_chromosome(
        chromosome,
        didp_model_registry,
        dual_bound_expression_function
    )

    # 3. Calculate Deviation and Fitness
    # Handle Solver Failure (Infeasible or Error)
    if solver_result_cost == float('inf'):
        fitness = float('inf')
    
    else:
        # Calculate deviation from the reference (optimal) point
        if reference_point != 0:
            deviation = abs(solver_result_cost - reference_point) / abs(reference_point)
        else:
            deviation = abs(solver_result_cost - reference_point)

        # 4. Apply Penalties based on Constraints
        # CONSTRAINT: The solver cost should match the reference (Optimal).
        # If Solver Cost > Reference: Heuristic was likely inadmissible (pruned the optimal path).
        # If Solver Cost == Reference: Heuristic was safe. Fitness is based on deviation (0) or efficiency.
        
        tolerance = 1e-6 # For float comparison
        
        if solver_result_cost <= reference_point + tolerance:
            # Case 1: Success (Optimal Solution Found)
            # Fitness is the deviation (ideally 0). 
            # You could add logic here to reward fewer expanded nodes if available.
            fitness = deviation
        else:
            # Case 2: Failure (Suboptimal Solution Found)
            # The heuristic likely overestimated and pruned the optimal path.
            # Apply massive penalty.
            fitness = deviation + 10000.0

    # 5. Update and Return
    chromosome_fitness_dict['fitness'] = fitness
    
    return chromosome_fitness_dict

##### *Testing fitness calculation of a chromosome*

In [20]:
def create_cvrp_didp_model():
    n = 4
    m = 2
    q = 5
    # Weights (demand)
    d = [0, 2, 3, 3]
    # Distance matrix
    distance_list = [
        [0, 3, 4, 5],
        [3, 0, 5, 4],
        [4, 5, 0, 3],
        [5, 4, 3, 0]
    ]
    distance_matrix_np = np.array(distance_list)
    model = m_dp.Model()
    customer = model.add_object_type(number=n)
    unvisited_var = model.add_set_var(object_type=customer, target=list(range(1, n)), name='unvisited_customers')
    location_var = model.add_element_var(object_type=customer, target=0)
    load_var = model.add_int_resource_var(target=0, less_is_better=True)
    vehicles_var = model.add_int_resource_var(target=1, less_is_better=True)
    weight = model.add_int_table(d)
    distance_table = model.add_int_table(distance_list)
    model.add_base_case([unvisited_var.is_empty(), location_var == 0])
    for j in range(1, n):
        visit = m_dp.Transition(
            name=f"visit {j}",
            cost=distance_table[location_var, j] + m_dp.IntExpr.state_cost(),
            effects=[
                (unvisited_var, unvisited_var.remove(j)),
                (location_var, j),
                (load_var, load_var + weight[j]),
            ],
            preconditions=[unvisited_var.contains(j), load_var + weight[j] <= q],
        )
        model.add_transition(visit)
    for j in range(1, n):
        visit_via_depot = m_dp.Transition(
            name=f"visit {j} with new vehicle",
            cost=distance_table[location_var, 0] + distance_table[0, j] + m_dp.IntExpr.state_cost(),
            effects=[
                (unvisited_var, unvisited_var.remove(j)),
                (location_var, j),
                (load_var, weight[j]),
                (vehicles_var, vehicles_var + 1),
            ],
            preconditions=[unvisited_var.contains(j), vehicles_var < m],
        )
        model.add_transition(visit_via_depot)
    return_to_depot = m_dp.Transition(
        name="return",
        cost=distance_table[location_var, 0] + m_dp.IntExpr.state_cost(),
        effects=[(location_var, 0)],
        preconditions=[unvisited_var.is_empty(), location_var != 0],
    )
    model.add_transition(return_to_depot)
    model.add_state_constr((m - vehicles_var + 1) * q - load_var >= weight[unvisited_var])
    # Return vars needed for heuristics
    didp_model = model
    didp_model_metadata_dict = {
        "unvisited": unvisited_var,
        "load": load_var, 
        "capacity": q}
    didp_bundle = didp_model, didp_model_metadata_dict
    return didp_bundle

def cvrp_dual_bound_expression_function(didp_bundle):
    """
    Creates the base heuristics linked to the current model instance.
    """
    didp_model, didp_model_metadata_dict = didp_bundle
    # Extract variables for easy access
    unvisited = didp_model_metadata_dict['unvisited']
    load = didp_model_metadata_dict['load']
    capacity = didp_model_metadata_dict['capacity']

    # --- Define Base Bounds ---
    
    def h1(state):
        # Logic: If load is high, we might need a return trip
        current_load = state[load]
        if current_load > (capacity * 0.8):
            return 5.0
        return 0.0

    def h2(state):
        # Logic: Cost is at least the number of unvisited nodes
        return float(len(state[unvisited]))

    def h3(state):
        return 1.0

    # Return the registry dictionary
    dual_bound_dict = automatic_creation_of_dual_bounds_registry(locals())
    #Manual ways: dual_bound_dict = {'h1': h1, 'h_unv': h2, 'h3': h3}
    return dual_bound_dict
    
# 1. Define an individual (Chromosome Dict)
chromosome_fitness_dict = {
    "chromosome": ["h1", "h2", "MAX", "h3", "ADD"], 
    "fitness": None
}

# 2. Define Reference Point (Optimal Cost)
# Assume we know the optimal cost for this small instance is 10
OPTIMAL_COST_REFERENCE = 1000

print(f"Initial Individual: {chromosome_fitness_dict}")

# 3. Evaluate Fitness
evaluated_individual = chromosome_fitness_dict_evaluation(
    chromosome_fitness_dict,
    create_cvrp_didp_model,
    cvrp_dual_bound_expression_function,
    OPTIMAL_COST_REFERENCE
)

print(f"\nEvaluated Individual: {evaluated_individual}")

# Output Interpretation:
# If fitness is ~0.0, the heuristic found the optimal solution (10.0).
# If fitness is > 10000, the heuristic was inadmissible and found a worse solution (or none).

Initial Individual: {'chromosome': ['h1', 'h2', 'MAX', 'h3', 'ADD'], 'fitness': None}

Evaluated Individual: {'chromosome': ['h1', 'h2', 'MAX', 'h3', 'ADD'], 'fitness': 0.98}


# **5. Initialization**

In [21]:
def initialize_list_of_chromosome_fitness_dictionary(list_size, dual_bound_functions_dict, 
                                                     LB_range_of_constant, UB_range_of_constant,
                                                     didp_model_registry, dual_bound_expression_function,
                                                     min_chromosome_length = min_chromosome_length,
                                                     max_chromosome_length = max_chromosome_length,
                                                     available_operations = available_operations,
                                                     reference_point = OPTIMAL_COST_REFERENCE):
    """
    Creates a population list where each element is a dictionary holding the chromosome 
    AND its evaluated fitness.
    
    Args:
        list_size (int): Number of individuals to generate.
        didp_model_registry (func): Factory for the DIDP model.
        dual_bound_expression_function (func): Factory for the heuristics.
        reference_point (float): Optimal cost for fitness calculation.
        [Other args match the generator inputs]

    Returns:
        list: A list of fully evaluated dicts: [{'chromosome': [...], 'fitness': 0.5}, ...]
    """
    population = []
    
    for _ in range(list_size):
        # 1. Generate Chromosome
        newly_generated_chromosome_fitness_dict = generate_combined_chromosome(
            dual_bound_functions_dict, 
            LB_range_of_constant, 
            UB_range_of_constant,
            min_chromosome_length, 
            max_chromosome_length, 
            available_operations
        )

        
        # 3. Evaluate Fitness Immediately
        evaluated_newly_generated_chromosome_fitness_dict = chromosome_fitness_dict_evaluation(
            newly_generated_chromosome_fitness_dict, 
            didp_model_registry, 
            dual_bound_expression_function, 
            reference_point
        )
        
        population.append(evaluated_newly_generated_chromosome_fitness_dict)

    return population

#### *Testing initialization list of chromosome_fitnes dictionary*

In [22]:
Test_OPTIMAL_COST_REFERENCE = 5 
def creation_of_didp_model_function():
    import modified_didppy as m_dp
    # [Data definition]
    
    #[DIDP Model Creation]
    model = m_dp.Model()
    # Create an integer variable 'x'. Set the START STATE to 5.
    x = model.add_int_var(target=5)
    # Set the GOAL CONDITION to x == 0.
    model.add_base_case([x == 0])
    #
    # --- END OF FIX ---
    #
    model.add_transition(
        m_dp.Transition(
            name="decrement",
            cost=1 + m_dp.IntExpr.state_cost(), # Cost is 1 per step
            effects=[(x, x - 1)]
        )
    )
    # This line is not needed for forward search, so we remove it.
    # model.target_state[x] = 5 
    # 2. Add a standard Rust expression bound
    model.add_dual_bound(0) 
    # [Return variables needed for dual bounds calculation]
    didp_model = model 
    didp_model_metadata_dict = {"x": x}
    didp_bundle = (didp_model, didp_model_metadata_dict)
    return didp_bundle

def dual_bound_expression_function(didp_bundle):
    didp_model, didp_model_metadata_dict = didp_bundle
    # [Variables/data definition]
    x = didp_model_metadata_dict['x']
    #[Dual bound logic declaration]
    def h1(state):
        return state[x]+1 # Example logic
    def h2(state):
        return 20
    def h3(state):
        return 5
    # [Return a dual bound registry dict]
    dual_bound_dict = automatic_creation_of_dual_bounds_registry(locals())
    return dual_bound_dict
available_operations = ["ADD", "SUBTRACT", "MAX", "MIN", "MULTIPLY", "PDIV"]

dual_bound_functions_registry = dual_bound_expression_function(creation_of_didp_model_function())

list_size = 5

population = initialize_list_of_chromosome_fitness_dictionary(
    list_size, dual_bound_functions_registry, 
    LB_range_of_constant, UB_range_of_constant,
    didp_model_registry = creation_of_didp_model_function,
    dual_bound_expression_function = dual_bound_expression_function,
    min_chromosome_length = min_chromosome_length,
    max_chromosome_length = max_chromosome_length,
    available_operations = available_operations,
    reference_point = Test_OPTIMAL_COST_REFERENCE
)

print(population)
print(f"Generated population of size {len(population)}:")
for i, individual in enumerate(population):
    print(f"Individual {i+1}:")
    print(f"  Chromosome: {individual['chromosome']}")
    print(f"  Fitness: {individual['fitness']}")
    # Optionally, compile and test each chromosome to ensure it's valid
    try:
        compiled_func = compile_chromosome_to_useable_function(individual, dual_bound_functions_registry, print_code=True)
    except Exception as e:
        print(f"  Error compiling or executing chromosome: {e}")

[{'chromosome': ['h3', 'h1', 9.97, 'h2', 'MULTIPLY', 'PDIV', 'SUBTRACT'], 'fitness': 0.0}, {'chromosome': ['h1', 'h2', 7.7, 'h3', 'MULTIPLY', 'MULTIPLY', 'MULTIPLY'], 'fitness': 0.0}, {'chromosome': ['h3', 'h1', 'h2', 'SUBTRACT', 'MIN'], 'fitness': 0.0}, {'chromosome': ['h3', 5.89, 'h1', 'MULTIPLY', 0.9, 'h2', 'MULTIPLY', 'ADD', 'PDIV'], 'fitness': 0.0}, {'chromosome': ['h1', 'h2', 'ADD', 3.45, 'h2', 'MULTIPLY', 7.33, 'h1', 'MULTIPLY', 'MAX', 'MAX', 2.31, 'h3', 'MULTIPLY', 6.11, 'h3', 'MULTIPLY', 'PDIV', 4.37, 'h3', 'MULTIPLY', 'h1', 'MIN', 'MAX', 'MIN', 1.25, 'h1', 'MULTIPLY', 8.17, 'h3', 'MULTIPLY', 'PDIV', 4.35, 'h2', 'MULTIPLY', 'h3', 'ADD', 'MULTIPLY', 8.81, 'h3', 'MULTIPLY', 8.74, 'h2', 'MULTIPLY', 'SUBTRACT', 'h1', 3.93, 'h3', 'MULTIPLY', 'PDIV', 'ADD', 'SUBTRACT', 'SUBTRACT', 'h1', 'h2', 'MULTIPLY', 5.94, 'h3', 'MULTIPLY', 'h1', 'MAX', 'PDIV', 6.48, 'h3', 'MULTIPLY', 'h2', 'MULTIPLY', 'h3', 2.99, 'h1', 'MULTIPLY', 'ADD', 'MIN', 'MAX', 'h1', 4.81, 'h1', 'MULTIPLY', 'MIN', 'h3', 

# **6. Parent selection operator**

In [23]:
def parents_selection(population, tournament_size=random.randint(2, 10), tournament_probability=0.8):
    """
    Selects a single parent using a Tournament Selection.
    
    Features:
    - Random Tournament Size: Defaults to random(2, 10) if not specified.
    - While Loop Logic: Uses manual loops for selection and knockout as requested.
    """

        
    # Safety: Cannot have a tournament larger than the population
    actual_size = min(len(population), tournament_size)
    
    # 1. Select Candidates (Using While Loop)
    list_of_candidates = []
    
    while len(list_of_candidates) < actual_size:
        candidate_index = random.randint(0, len(population) - 1)
        candidate = population[candidate_index]
        
        # Ensure uniqueness (simulating random.sample logic with a loop)
        # Note: In a list of dicts, simple equality checks work if they are the exact same object
        if candidate not in list_of_candidates:
            list_of_candidates.append(candidate)
            
    # 3. Apply Tournament Logic
    if random.random() < tournament_probability:
        winner_of_tournament = list_of_candidates.copy()
        
        # Knockout Loop: Fight until 1 remains
        while len(winner_of_tournament) > 1:
            # Pick two random fighters
            idx_1 = random.randint(0, len(winner_of_tournament) - 1)
            candidate_1 = winner_of_tournament[idx_1]
            winner_of_tournament.remove(candidate_1)
            
            idx_2 = random.randint(0, len(winner_of_tournament) - 1)
            candidate_2 = winner_of_tournament[idx_2]
            winner_of_tournament.remove(candidate_2)
            
            # Extract Fitness (Handle None as Infinity for minimization)
            f1 = candidate_1['fitness'] if candidate_1['fitness'] is not None else float('inf')
            f2 = candidate_2['fitness'] if candidate_2['fitness'] is not None else float('inf')
            
            # Compare (Lower Fitness is Better)
            if f1 < f2:
                winner_of_tournament.append(candidate_1)
            else:
                winner_of_tournament.append(candidate_2)
        
        # The champion
        parent = winner_of_tournament[0]
        
    else:
        # --- Random Mode (20% chance) ---
        parent = random.choice(list_of_candidates)
        
    return parent

#### *Testing parent selection*

In [24]:
# Mock Population
test_population = [
    {'chromosome': ['h2','h1', 'ADD'], 'fitness': 100.0}, # Bad
    {'chromosome': ['h2','h2', 'ADD'], 'fitness': 5.0},   # Good
    {'chromosome': ['h2','h3', 'ADD'], 'fitness': 50.0},  # Medium
    {'chromosome': ['h2','h4', 'ADD'], 'fitness': float('inf')}, # Invalid
]

print("--- Testing Parent Selection ---")

# Run 10 times to see behavior
for i in range(10):
    selected = parents_selection(test_population, tournament_probability=0.8)
    print(f"Run {i+1}: Selected Fitness {selected['fitness']}")

--- Testing Parent Selection ---
Run 1: Selected Fitness inf
Run 2: Selected Fitness 5.0
Run 3: Selected Fitness 5.0
Run 4: Selected Fitness 100.0
Run 5: Selected Fitness 50.0
Run 6: Selected Fitness 5.0
Run 7: Selected Fitness 5.0
Run 8: Selected Fitness 5.0
Run 9: Selected Fitness 5.0
Run 10: Selected Fitness 5.0


# **7. Crossover operator**

## 1-point crossover

### Helper functions of 1-point  crossover

In [34]:
def get_rpn_node_arity(gene, available_operations=available_operations):
    """
    Returns 2 for binary operators, 0 for terminals.
    Includes a check to ensure 'gene' is a string before checking the list.
    """
    # isinstance is a built-in Python function, it should always be available.
    # We check if it is a string to avoid errors if 'gene' is a number (float/int).
    if isinstance(gene, str) and gene in available_operations:
        return 2
    return 0

def construct_the_lookup_map_for_rpn(chromosome_rpn_topology):
    """
    Creates a map {end_index: (start, end, type, depth)}
    """
    lookup = {}
    for idx, node_data in chromosome_rpn_topology.items():
        start, end = node_data['subtree_range']
        node_type = node_data['type']
        depth = node_data['depth'] # <--- Retrieve it
        
        # Store as 4-element tuple
        lookup[end] = (start, end, node_type, depth)
        
    return lookup

def map_chromosome_fitness_dict_to_rpn_topology(chromosome_fitness_dict, available_operations=available_operations):
    """
    Traverses RPN *Forwards* to map indices to tree structure.
    
    Why Forward?
    RPN is 'Left Child' -> 'Right Child' -> 'Operator'.
    By scanning forward, we ensure children are already on the stack 
    when we hit an operator.
    
    Returns a dict: index -> {'type', 'subtree_range', 'left_child', 'right_child'}
    """
    topology = {}
    stack = []
    chromosome = extract_chromosome_from_chromosome_fitness_dict(chromosome_fitness_dict)
    for i, gene in enumerate(chromosome):
        arity = get_rpn_node_arity(gene, available_operations)
        node_value = gene 
        
        if arity == 0:
            subtree_start_location = i
            stack.append((i, subtree_start_location))
            
            topology[i] = {
                'type': 'TERMINAL', 
                'value': node_value, 
                'subtree_range': (i, i),
                'depth': len(stack)  # Depth is current stack size
            }
            
        elif arity == 2:
            if len(stack) < 2: return {} 
            
            right_idx, right_start = stack.pop()
            left_idx, left_start = stack.pop()
            subtree_range = (left_start, i)
            current_depth = len(stack) + 1
            
            topology[i] = {
                'type': 'FUNCTION', 
                'value': node_value,
                'subtree_range': subtree_range,
                'left_child': left_idx,
                'right_child': right_idx,
                'depth': current_depth
            }
            stack.append((i, left_start))
            
    return topology

def find_homologous_pairs(parent1_dict, parent2_dict, available_operations=available_operations):
    """
    Identifies indices in Parent 1 and Parent 2 that share the same topology 
    by traversing both trees from the Root down.
    """
    # 1. Extract raw lists
    p1_chromosome = extract_chromosome_from_chromosome_fitness_dict(parent1_dict)
    p2_chromosome = extract_chromosome_from_chromosome_fitness_dict(parent2_dict)
    
    # 2. Build Topology Maps (Index -> Node Data)
    # FIX: Call the correct mapping function first!
    p1_topology = map_chromosome_fitness_dict_to_rpn_topology(parent1_dict, available_operations)
    p2_topology = map_chromosome_fitness_dict_to_rpn_topology(parent2_dict, available_operations)
    
    matching_pairs = []
    
    # 3. Initialize Traversal Queue (Start at Roots)
    root_p1 = len(p1_chromosome) - 1
    root_p2 = len(p2_chromosome) - 1
    
    queue = [(root_p1, root_p2)]
    
    # 4. BFS Traversal
    while queue:
        idx1, idx2 = queue.pop(0)
        
        # Retrieve node structure data
        node1 = p1_topology.get(idx1)
        node2 = p2_topology.get(idx2)
        
        if not node1 or not node2: 
            continue
        
        # Check Homology
        if node1['type'] == node2['type']:
            matching_pairs.append((idx1, idx2))
            
            if node1['type'] == 'FUNCTION':
                queue.append((node1['left_child'], node2['left_child']))
                queue.append((node1['right_child'], node2['right_child']))
                
    return matching_pairs, p1_topology, p2_topology

### Core functions of 1-point crossover

In [ ]:
def one_point_crossover_two_offspring(parent1_dict, parent2_dict, dual_bound_functions_registry, 
                                      homology_1_point_crossover_probability= homology_1_point_crossover_probability):
    """
    Performs 1-Point Crossover.
    Priority: Try to swap at a Homologous Point (Matching Structure).
    Fallback: If no homology, swap at random Function points in both parents.
    Swaps the PREFIX (Head) of the chromosomes.
    """
    p1_chromosome = extract_chromosome_from_chromosome_fitness_dict(parent1_dict)
    p2_chromosome = extract_chromosome_from_chromosome_fitness_dict(parent2_dict)
    
    # 1. Find Homologous Pairs (Common Regions) using the Helper
    matching_pairs, p1_topology, p2_topology= find_homologous_pairs(parent1_dict, parent2_dict)

    # 2. Select Crossover Points
    if matching_pairs and random.random() < homology_1_point_crossover_probability:
        # CASE A: Homology Found - Pick matching structure
        cp1, cp2 = random.choice(matching_pairs)
    else:
        # CASE B: Fallback - Random Functions with MATCHING DEPTH
        p1_lookup = construct_the_lookup_map_for_rpn(p1_topology)
        p2_lookup = construct_the_lookup_map_for_rpn(p2_topology)
        
        # Filter: Must be FUNCTION
        # lookup items: (start, end, type, depth)
        p1_funcs = [idx for idx, data in p1_lookup.items() if data[2] == 'FUNCTION']
        p2_funcs = [idx for idx, data in p2_lookup.items() if data[2] == 'FUNCTION']
        
        # Find compatible pairs (Same Stack Depth)
        compatible_pairs = []
        for i1 in p1_funcs:
            depth1 = p1_lookup[i1][3] # Index 3 is Depth
            for i2 in p2_funcs:
                depth2 = p2_lookup[i2][3]
                if depth1 == depth2:
                    compatible_pairs.append((i1, i2))
    
        if compatible_pairs:
            cp1, cp2 = random.choice(compatible_pairs)
        else:
            # Absolute fallback: Roots always have depth 1
            cp1 = len(p1_chromosome) - 1
            cp2 = len(p2_chromosome) - 1
    
    # 3. Perform Prefix Swap (Head Swap)
    # Offspring 1: Head of P2 + Tail of P1
    offspring1 = p2_chromosome[:cp2+1] + p1_chromosome[cp1+1:]
    
    # Offspring 2: Head of P1 + Tail of P2
    offspring2 = p1_chromosome[:cp1+1] + p2_chromosome[cp2+1:]
    
    # 4. Validate & Compile
    offspring_list = []
    fallback_candidate_list  = [p2_chromosome, p1_chromosome]
    for i, chromosome in enumerate([offspring1, offspring2]):
        chromo_fitness_dict = {'chromosome': chromosome, 'fitness': 0}
        try:
            func = compile_chromosome_to_useable_function(chromo_fitness_dict, dual_bound_functions_registry, print_code=False)
            offspring_list.append(chromosome)
        except Exception as e:
            print(f"Offspring {i+1} Compilation Failed: {e}")
            # Optionally, could append None or a default value
            fallback_candidate = random.choice(fallback_candidate_list)
            fallback_candidate_list.remove(fallback_candidate)
            print(f"Using fallback parent chromosome for Offspring {i+1}.")
            offspring_list.append(fallback_candidate)
    return offspring_list

##### *Testing 1-point crossover*

In [36]:
def h1(state): return sum(state)
def h2(state): return 20
def h3(state): return 5

dual_bound_functions_registry = {
    "h1": h1,
    "h2": h2,
    "h3": h3
}
operations = ["ADD", "SUBTRACT", "MAX", "MIN", "MULTIPLY"]
# 1. Define Parents as per your example
# P1: [a1, h1, multiply, h2, add]
# Note: using strings for a1/h1 for visualization
parent1 = {
    'chromosome': [2, 'h1', 'MULTIPLY', 'h2', 'ADD'], # a1=2 for clarity
    'fitness': None
}

# P2: [h1, h2, multiply, h3, add]
parent2 = {
    'chromosome': [3,'h1','MULTIPLY', 4, 'h2','MULTIPLY', 'MULTIPLY', 'h3', 'ADD'],
    'fitness': None
}

print(f"Parent 1: {parent1['chromosome']}")
print(f"Parent 2: {parent2['chromosome']}")

# 2. Perform Crossover
child_chromosome_list = one_point_crossover_two_offspring(parent1, parent2, dual_bound_functions_registry)

Parent 1: [2, 'h1', 'MULTIPLY', 'h2', 'ADD']
Parent 2: [3, 'h1', 'MULTIPLY', 4, 'h2', 'MULTIPLY', 'MULTIPLY', 'h3', 'ADD']
Generated Code:
def dual_bound_combination(state):
    return (((3 * h1(state)) * (4 * h2(state))) + h3(state))

Generated Code:
def dual_bound_combination(state):
    return ((2 * h1(state)) + h2(state))



## Subtree crossover

### Core functions

In [ ]:
def subtree_crossover(parent1_dict, parent2_dict, dual_bound_functions_registry, subtree_crossover_probability=subtree_crossover_probability, available_operations=available_operations):
    """
    Performs subtree crossover between two parents.
    Input: Dicts {'chromosome': [...], 'fitness': ...}
    Output: A new child chromosome (list).
    """
    p1_chromosome = extract_chromosome_from_chromosome_fitness_dict(parent1_dict)
    p2_chromosome = extract_chromosome_from_chromosome_fitness_dict(parent2_dict)
    
    # --- Step 1: Identify Structures ---
    p1_structure = analyzing_chromosome_based_on_rpn_structure(parent1_dict)
    p2_structure = analyzing_chromosome_based_on_rpn_structure(parent2_dict)
    
    # --- Step 2: Choose Point for Parent 1 (The Receiver) ---
    # We decide whether we want to replace a Function or a Terminal in P1
    if random.random() < subtree_crossover_probability and p1_structure["FUNCTIONS"]:
        # 90% chance: Replace a Function tree
        candidate1 = p1_structure["FUNCTIONS"]
    else:
        # 10% chance: Replace a Terminal leaf
        candidate1 = p1_structure["TERMINALS"]    
    # Fallback: if P1 has no functions (it's just a leaf), must pick terminal
    # Filter out the root (which is always the last index in RPN)
    root_index = len(p1_chromosome) - 1
    valid_candidates = [
        (s, e) for (s, e) in candidate1 
        if e != root_index  # Don't let the end of the subtree be the root
    ]

    if valid_candidates:
        p1_start, p1_end = random.choice(valid_candidates)
    else:
        # Fallback: If no other options exist (e.g., P1 is already just a leaf), 
        # you might have to pick the root or skip crossover.
        candidate1 = p1_structure["TERMINALS"]   
        p1_start, p1_end = random.choice(candidate1)
    
    # --- Step 3: Choose Point for Parent 2 (The Donor) ---
    # The donor part can be anything (Function or Terminal), 
    # as long as it produces 1 value.
    if random.random() < subtree_crossover_probability and p2_structure["FUNCTIONS"]:
        # 90% chance: Replace a Function tree
        candidate2 = p2_structure["FUNCTIONS"]
    else:
        # 10% chance: Replace a Terminal leaf
        candidate2 = p2_structure["TERMINALS"]  
    if not candidate2: candidate2 = p2_structure["TERMINALS"]   
    p2_start, p2_end = random.choice(candidate2)
    
    # --- Step 4: Process Crossover ---
    # Construct Offspring: P1_before_crossover_point + P2_Subtree + P1_after_crossover_point
    
    # 1. Part of P1 before the cut
    head_of_parent1_chromosome = p1_chromosome[:p1_start]
    
    # 2. The subtree from P2
    donor_gene = p2_chromosome[p2_start : p2_end+1]
    
    # 3. Part of P1 after the cut
    tail_of_parent1_chromosome = p1_chromosome[p1_end+1:]
    
    offspring_chromosome = head_of_parent1_chromosome + donor_gene + tail_of_parent1_chromosome
    
    # --- Step 5: Safety validation ---
    try:
        offspring_chromosome_function = compile_chromosome_to_useable_function(
            {'chromosome': offspring_chromosome, 'fitness':0}, 
            dual_bound_functions_registry, 
            print_code=False
        )
    except Exception as e:
        print(f"Crossover produced invalid offspring: {e}")
        # In case of invalid offspring, return parent1's chromosome as fallback
        return random.choice([parent1_dict['chromosome'], parent2_dict['chromosome']])

    return offspring_chromosome

#### *Testing subtree crossover*

In [38]:
def h1(state): return sum(state)
def h2(state): return 20
def h3(state): return 5

dual_bound_functions_registry = {
    "h1": h1,
    "h2": h2,
    "h3": h3
}
operations = ["ADD", "SUBTRACT", "MAX", "MIN", "MULTIPLY"]
# 1. Define Parents as per your example
# P1: [a1, h1, multiply, h2, add]
# Note: using strings for a1/h1 for visualization
parent1 = {
    'chromosome': [2, 'h1', 'MULTIPLY', 'h2', 'ADD'], # a1=2 for clarity
    'fitness': None
}

# P2: [h1, h2, multiply, h3, add]
parent2 = {
    'chromosome': [3,'h1','MULTIPLY', 4, 'h2','MULTIPLY', 'MULTIPLY', 'h3', 'ADD'],
    'fitness': None
}

print(f"Parent 1: {parent1['chromosome']}")
print(f"Parent 2: {parent2['chromosome']}")

# 2. Perform Crossover
child_chromosome = subtree_crossover(parent1, parent2, dual_bound_functions_registry)

Parent 1: [2, 'h1', 'MULTIPLY', 'h2', 'ADD']
Parent 2: [3, 'h1', 'MULTIPLY', 4, 'h2', 'MULTIPLY', 'MULTIPLY', 'h3', 'ADD']
Generated Code:
def dual_bound_combination(state):
    return ((2 * h1(state)) + (((3 * h1(state)) * (4 * h2(state))) + h3(state)))



## Uniform crossover

In [ ]:
def uniform_crossover_weighted_protected(parent1_dict, parent2_dict, dual_bound_functions_registry, uniform_crossover_probability=uniform_crossover_probability):
    """
    Performs Uniform Crossover with Atomic Block Swapping.
    
    - Standard: Swaps compatible genes (Term<->Term, Func<->Func).
    - Protected Blocks: If BOTH parents have a [Num, Str, MUL] block at the same spot,
      swaps the ENTIRE block as a single unit (Safer).
    - Unmatched Blocks: Protects the 'MULTIPLY' operator but allows constituents to swap.
    """
    p1_chromosome = extract_chromosome_from_chromosome_fitness_dict(parent1_dict)
    p2_chromosome = extract_chromosome_from_chromosome_fitness_dict(parent2_dict)
    
    # 1. Analyze Structure
    p1_struct = analyzing_chromosome_based_on_rpn_structure(parent1_dict)
    p2_struct = analyzing_chromosome_based_on_rpn_structure(parent2_dict)
    
    # 2. Build Sets of "Protected Operators" (The MULTIPLY index)
    protected_indices_p1 = set()
    for start, end in p1_struct["TERMINALS"]:
        if (end - start) == 2: 
            protected_indices_p1.add(end)
            
    protected_indices_p2 = set()
    for start, end in p2_struct["TERMINALS"]:
        if (end - start) == 2:
            protected_indices_p2.add(end)
            
    # 3. Iteration Setup
    min_len = min(len(p1_chromosome), len(p2_chromosome))
    offspring1 = p1_chromosome.copy()
    offspring2 = p2_chromosome.copy()
    
    i = 0
    # Stop before the Root (last index) to protect it
    while i < min_len - 1:
        
        # --- LOGIC 1: ATOMIC BLOCK SWAP (The "Safer" Logic) ---
        # Check if we are at the start of a Protected Block in BOTH parents.
        # A block [Num, Str, MUL] starting at 'i' ends at 'i+2'.
        block_end_index = i + 2
        
        # Ensure we don't go out of bounds
        if block_end_index < (min_len - 1):
            is_block_p1 = (block_end_index in protected_indices_p1)
            is_block_p2 = (block_end_index in protected_indices_p2)
            
            if is_block_p1 and is_block_p2:
                # Both have a full TERMINAL block. Swap the WHOLE BLOCK as one unit.
                if random.random() < uniform_crossover_probability:
                    # Swap Weight (i)
                    offspring1[i], offspring2[i] = offspring2[i], offspring1[i]
                    # Swap Heuristic (i+1)
                    offspring1[i+1], offspring2[i+1] = offspring2[i+1], offspring1[i+1]
                    # Swap Operator (i+2) - They are identical MULs, but good for consistency
                    offspring1[i+2], offspring2[i+2] = offspring2[i+2], offspring1[i+2]
                
                # Advance 3 steps (Skip the constituents we just handled)
                i += 3
                continue
        # --- LOGIC 2: STANDARD CONSTITUENT SWAP ---
        gene1 = p1_chromosome[i]
        gene2 = p2_chromosome[i]
        
        arity1 = get_rpn_node_arity(gene1)
        arity2 = get_rpn_node_arity(gene2)
        
        # Only swap if Arity matches
        if arity1 == arity2:
            # Protection Check: Don't swap if one is a Protected Operator and the other isn't
            is_protected_p1 = (i in protected_indices_p1)
            is_protected_p2 = (i in protected_indices_p2)
            
            if (is_protected_p1 or is_protected_p2) and (gene1 != gene2):
                pass # Block the swap
            else:
                # Standard Swap (Individual genes)
                if random.random() < uniform_crossover_probability:
                    offspring1[i] = gene2
                    offspring2[i] = gene1
        i += 1
                
    # 4. Validation & Return
    offspring_list = []
    for chromosome, original in [(offspring1, p1_chromosome), (offspring2, p2_chromosome)]:
        try:
            compile_chromosome_to_useable_function({'chromosome': chromosome, 'fitness':0}, dual_bound_functions_registry, print_code=False)
            offspring_list.append(chromosome)
        except Exception as e:
            print(f"Offspring Compilation Failed: {e}")
            print(f"Using fallback parent chromosome.")
            offspring_list.append(original)
            
    return offspring_list

#### *Testing uniform crossover*

In [40]:
def h1(state): return sum(state)
def h2(state): return 20
def h3(state): return 5

dual_bound_functions_registry = {
    "h1": h1,
    "h2": h2,
    "h3": h3
}
operations = ["ADD", "SUBTRACT", "MAX", "MIN", "MULTIPLY"]
# 1. Define Parents as per your example
# P1: [a1, h1, multiply, h2, add]
# Note: using strings for a1/h1 for visualization
parent1 = {
    'chromosome': [2, 'h1', 'MULTIPLY', 'h2', 'ADD'], # a1=2 for clarity
    'fitness': None
}

# P2: [h1, h2, multiply, h3, add]
parent2 = {
    'chromosome': [3,'h1','MULTIPLY', 4, 'h2','MULTIPLY', 'MULTIPLY', 'h3', 'ADD'],
    'fitness': None
}

print(f"Parent 1: {parent1['chromosome']}")
print(f"Parent 2: {parent2['chromosome']}")

# 2. Perform Crossover
child_chromosome = uniform_crossover_weighted_protected(parent1, parent2, dual_bound_functions_registry)

Parent 1: [2, 'h1', 'MULTIPLY', 'h2', 'ADD']
Parent 2: [3, 'h1', 'MULTIPLY', 4, 'h2', 'MULTIPLY', 'MULTIPLY', 'h3', 'ADD']
Generated Code:
def dual_bound_combination(state):
    return ((2 * h1(state)) + h2(state))

Generated Code:
def dual_bound_combination(state):
    return (((3 * h1(state)) * (4 * h2(state))) + h3(state))



## Combined crossover generator

In [56]:
def combined_crossover_generator(parent1_dict, parent2_dict, dual_bound_functions_registry, 
                                 didp_model_registry, dual_bound_expression_function, 
                                 reference_point = OPTIMAL_COST_REFERENCE, 
                                 homology_1_point_crossover_probability=homology_1_point_crossover_probability, 
                                 subtree_crossover_probability=subtree_crossover_probability,
                                 uniform_crossover_probability=uniform_crossover_probability,
                                 available_operations=available_operations):
    """
    Randomly selects one of the three crossover methods to produce offspring,
    then evaluates the fitness of the resulting offspring.
    
    Returns:
        list: A list of evaluated dictionaries [{'chromosome': [...], 'fitness': float}, ...]
    """
    crossover_methods = [
        "one_point",
        "subtree",
        "uniform"
    ]
    
    # 1. Select Method
    selected_method = random.choice(crossover_methods)
    
    raw_offspring_chromosomes = []
    
    # 2. Generate Offspring (Raw Lists)
    if selected_method == "one_point":
        # print("Using One-Point Crossover with Homology.")
        raw_offspring_chromosomes = one_point_crossover_two_offspring(
            parent1_dict, parent2_dict, dual_bound_functions_registry, homology_1_point_crossover_probability=homology_1_point_crossover_probability
        )
    
    elif selected_method == "subtree":
        # print("Using Subtree Crossover.")
        single_offspring = subtree_crossover(
            parent1_dict, parent2_dict, dual_bound_functions_registry, subtree_crossover_probability=subtree_crossover_probability
        )
        # Wrap the single result in a list to maintain consistency
        raw_offspring_chromosomes = [single_offspring]
    
    elif selected_method == "uniform":
        # print("Using Uniform Crossover with Weighted Protection.")
        raw_offspring_chromosomes = uniform_crossover_weighted_protected(
            parent1_dict, parent2_dict, dual_bound_functions_registry,  uniform_crossover_probability=uniform_crossover_probability,
        )

    # 3. Evaluate Fitness for All Offspring
    evaluated_offspring_list = []
    
    for offspring_chromosome in raw_offspring_chromosomes:
        # Create the dictionary structure
        offspring_chromosome_fitness_dict = {'chromosome': offspring_chromosome, 'fitness': None}
        
        # Calculate fitness
        evaluated_individual = chromosome_fitness_dict_evaluation(
            offspring_chromosome_fitness_dict, 
            didp_model_registry, 
            dual_bound_expression_function, 
            reference_point
        )
        
        evaluated_offspring_list.append(evaluated_individual)
        
    return evaluated_offspring_list

#### *Testing calculation of evaluated offspring*

In [60]:
Test_OPTIMAL_COST_REFERENCE = 5 
def creation_of_didp_model_function():
    import modified_didppy as m_dp
    # [Data definition]
    
    #[DIDP Model Creation]
    model = m_dp.Model()
    # Create an integer variable 'x'. Set the START STATE to 5.
    x = model.add_int_var(target=5)
    # Set the GOAL CONDITION to x == 0.
    model.add_base_case([x == 0])
    #
    # --- END OF FIX ---
    #
    model.add_transition(
        m_dp.Transition(
            name="decrement",
            cost=1 + m_dp.IntExpr.state_cost(), # Cost is 1 per step
            effects=[(x, x - 1)]
        )
    )
    # This line is not needed for forward search, so we remove it.
    # model.target_state[x] = 5 
    # 2. Add a standard Rust expression bound
    model.add_dual_bound(0) 
    # [Return variables needed for dual bounds calculation]
    didp_model = model 
    didp_model_metadata_dict = {"x": x}
    didp_bundle = (didp_model, didp_model_metadata_dict)
    return didp_bundle

def dual_bound_expression_function(didp_bundle):
    didp_model, didp_model_metadata_dict = didp_bundle
    # [Variables/data definition]
    x = didp_model_metadata_dict['x']
    #[Dual bound logic declaration]
    def h1(state):
        return state[x]+1 # Example logic
    def h2(state):
        return 20
    def h3(state):
        return 5
    # [Return a dual bound registry dict]
    dual_bound_dict = automatic_creation_of_dual_bounds_registry(locals())
    return dual_bound_dict
available_operations = ["ADD", "SUBTRACT", "MAX", "MIN", "MULTIPLY", "PDIV"]

dual_bound_functions_registry = dual_bound_expression_function(creation_of_didp_model_function())

parent1 = {
    'chromosome': [2, 'h1', 'MULTIPLY', 'h2', 'ADD'], # a1=2 for clarity
    'fitness': None
}

# P2: [h1, h2, multiply, h3, add]
parent2 = {
    'chromosome': [3,'h1','MULTIPLY', 4, 'h2','MULTIPLY', 'MULTIPLY', 'h3', 'ADD'],
    'fitness': None
}

print(f"Parent 1: {parent1['chromosome']}")
print(f"Parent 2: {parent2['chromosome']}")

print("--- Starting Combined Crossover Test ---")

# We run the generator. 
# It will:
# 1. Pick a crossover method (One Point, Subtree, or Uniform)
# 2. Generate offspring
# 3. Create a NEW didp model using 'creation_of_didp_model_function'
# 4. Link heuristics using 'dual_bound_expression_function'
# 5. Solve and calculate fitness
evaluated_offspring = combined_crossover_generator(
    parent1, 
    parent2, 
    dual_bound_functions_registry,
    creation_of_didp_model_function,   # Pass the model factory
    dual_bound_expression_function,    # Pass the heuristic factory
    reference_point = Test_OPTIMAL_COST_REFERENCE
)

# --- 5. Results ---
print(f"\nGenerated {len(evaluated_offspring)} offspring.")
for i, child in enumerate(evaluated_offspring):
    print(f"\nChild {i+1}:")
    print(f"  Chromosome: {child['chromosome']}")
    print(f"  Fitness:    {child['fitness']}")
    
    # Interpretation
    if child['fitness'] is not None and child['fitness'] < 1e-5:
        print("  -> Status: Optimal Solution Found (Fitness ~ 0)")
    elif child['fitness'] == float('inf'):
        print("  -> Status: Invalid/Infeasible")
    else:
        print(f"  -> Status: Valid but Suboptimal (Deviation: {child['fitness']})")

Parent 1: [2, 'h1', 'MULTIPLY', 'h2', 'ADD']
Parent 2: [3, 'h1', 'MULTIPLY', 4, 'h2', 'MULTIPLY', 'MULTIPLY', 'h3', 'ADD']
--- Starting Combined Crossover Test ---
Generated Code:
def dual_bound_combination(state):
    return ((2 * h1(state)) + 4)

Generated Code:
def dual_bound_combination(state):
    return (((3 * h1(state)) * (h2(state) * h2(state))) + h3(state))


Generated 2 offspring.

Child 1:
  Chromosome: [2, 'h1', 'MULTIPLY', 4, 'ADD']
  Fitness:    0.0
  -> Status: Optimal Solution Found (Fitness ~ 0)

Child 2:
  Chromosome: [3, 'h1', 'MULTIPLY', 'h2', 'h2', 'MULTIPLY', 'MULTIPLY', 'h3', 'ADD']
  Fitness:    0.0
  -> Status: Optimal Solution Found (Fitness ~ 0)


# **8. Mutation operator**

## Subtree mutation

In [ ]:
def subtree_mutation(chromosome_fitness_dict, dual_bound_functions_registry, LB_range_of_constant, UB_range_of_constant, 
                    mutation_max_subtree_depth = mutation_max_subtree_depth, available_operations = available_operations):
    """
    Performs Subtree Mutation.
    1. Selects a random node (subtree) in the parent chromosome.
    2. Generates a NEW random subtree (RPN list).
    3. Replaces the old subtree with the new one.
    
    Args:
        individual_dict (dict): Parent {'chromosome': [...], ...}
        mutation_max_depth (int): Max depth for the NEWLY generated subtree (usually small, e.g., 2-4).
        [Other args]: Standard generation parameters.
    """
    before_mutated_chromosome = extract_chromosome_from_chromosome_fitness_dict(chromosome_fitness_dict)
    
    # 1. Analyze Structure to find valid cut points
    # We reuse your existing helper to get all valid subtree ranges (Terminals & Functions)
    chromosome_structure = analyzing_chromosome_based_on_rpn_structure(chromosome_fitness_dict, available_operations)
    
    # Combine all possible cut points into one list
    # Each candidate is a tuple (start_index, end_index)
    candidates = chromosome_structure["FUNCTIONS"] + chromosome_structure["TERMINALS"]

    # 2. Select a Mutation Point
    cut_start, cut_end = random.choice(candidates)
    
    # 3. Generate a New Random Subtree
    # We generate a completely new valid RPN expression to graft in.
    # We use 'generate_ramped_half_and_half' to ensure diversity.
    new_subtree = generate_ramped_half_and_half(dual_bound_functions_registry, LB_range_of_constant, UB_range_of_constant,
        min_chromosome_length = min_chromosome_length, max_chromosome_length=mutation_max_subtree_depth)
    
    # 4. Grafting (Replace Old with New)
    # Prefix: Everything before the cut
    prefix = before_mutated_chromosome[:cut_start]
    
    # Suffix: Everything after the cut
    suffix = before_mutated_chromosome[cut_end+1:]
    
    # New Chromosome: Prefix + New_Subtree + Suffix
    mutated_chromosome = prefix + new_subtree + suffix
    
    # 5. Validation & Return
    try:
        # Check if valid RPN
        new_individual = {'chromosome': mutated_chromosome, 'fitness': 0}
        compile_chromosome_to_useable_function(new_individual, dual_bound_functions_registry, print_code=False)
        return mutated_chromosome
    except Exception as e:
        print(f"Mutation produced invalid tree: {e}")
        return before_mutated_chromosome # Fallback to original

#### *Testing subtree mutation*

In [44]:
def h1(state): return sum(state)
def h2(state): return 20
def h3(state): return 5

dual_bound_functions_registry = {
    "h1": h1,
    "h2": h2,
    "h3": h3
}

# 1. Define Parents as per your example
# P1: [a1, h1, multiply, h2, add]
# Note: using strings for a1/h1 for visualization
parent1 = {
    'chromosome': [2, 'h1', 'MULTIPLY', 'h2', 'ADD'], # a1=2 for clarity
    'fitness': None
}

print(f"Parent 1: {parent1['chromosome']}")

# 2. Perform Crossover
child_chromosome = subtree_mutation(parent1, dual_bound_functions_registry, LB_range_of_constant, UB_range_of_constant)

Parent 1: [2, 'h1', 'MULTIPLY', 'h2', 'ADD']
Generated Code:
def dual_bound_combination(state):
    return ((2 * h1(state)) + (max((((((min((5.86 * h1(state)), h2(state)) - max((0.76 * h1(state)), (7.36 * h1(state)))) - max((h3(state) * h1(state)), (h2(state) - h2(state)))) + ((min((4.74 * h2(state)), h2(state)) + min(h2(state), (6.78 * h2(state)))) + (min((1.98 * h3(state)), h2(state)) - ((5.25 * h3(state)) * (3.36 * h3(state)))))) - max(max((((2.87 * h2(state)) / (6.47 * h3(state)) if abs((6.47 * h3(state))) > 1e-6 else 1.0) + ((2.16 * h3(state)) + h2(state))), ((h1(state) * (1.38 * h1(state))) * min((6.06 * h1(state)), h3(state)))), ((max(h2(state), h3(state)) + (h2(state) * h2(state))) + min((h1(state) / (4.25 * h3(state)) if abs((4.25 * h3(state))) > 1e-6 else 1.0), (h3(state) + h2(state)))))) - (((((h2(state) + (2.88 * h1(state))) - ((5.86 * h3(state)) * (5.27 * h3(state)))) * max((h3(state) * (2.66 * h3(state))), (h3(state) + h1(state)))) / (max(((0.77 * h1(state)) - h3(state)),

## Point mutation

In [ ]:
def point_mutation(chromosome_fitness_dict, dual_bound_functions_registry, LB_range_of_constant, 
                   UB_range_of_constant, available_operations= available_operations):
    """
    Performs Point Mutation (Bit-Flip equivalent).
    Selects a random node and replaces it with a valid alternative of the same arity.
    
    Constraints enforced:
    1. Weighted Blocks [Num, Str, MUL]: 
       - Num (Coefficient): Can be altered.
       - Str (Heuristic): Can be altered.
       - MUL (Operator): PROTECTED (Cannot be altered).
    2. Alone Coefficients/Heuristics: Can be altered.
    3. Functions: Can be altered (e.g., ADD -> SUBTRACT).
    """
    before_mutation_chromosome = extract_chromosome_from_chromosome_fitness_dict(chromosome_fitness_dict)
    
    # 1. Analyze Structure to identify "Protected" nodes
    # We reuse your existing analyzer to find Weighted Blocks
    chromosome_structure = analyzing_chromosome_based_on_rpn_structure(chromosome_fitness_dict, available_operations)
    
    protected_indices = set()
    
    # Identify the 'MULTIPLY' at the end of Weighted Blocks [Num, Str, MUL]
    # The user rule: "if we reached a point belong to a terminal set... dont altered the 'multiply'"
    for start, end in chromosome_structure["TERMINALS"]:
        if (end - start) == 2: # Length 3 block [Num, Str, MUL]
            protected_indices.add(end) # The 'end' index is the protected MULTIPLY
            
    # 2. Identify Valid Mutation Candidates
    # All indices are valid EXCEPT the protected 'MULTIPLY's
    valid_indices = [i for i in range(len(before_mutation_chromosome)) if i not in protected_indices]
    
    if not valid_indices:
        return chromosome_fitness_dict # No mutation possible
        
    # 3. Select Random Node to Mutate
    mutation_idx = random.choice(valid_indices)
    original_gene = before_mutation_chromosome[mutation_idx]
    
    new_gene = original_gene # Default fallback
    
    # 4. Perform Mutation based on Node Type
    
    # --- CASE A: Coefficient (Float/Int) ---
    # "only the coeficients... can be altered"
    if isinstance(original_gene, (int, float)):
        # Generate a new random coefficient
        new_gene = round(random.uniform(LB_range_of_constant, UB_range_of_constant), 2)
        
    # --- CASE B: String (Heuristic or Operator) ---
    elif isinstance(original_gene, str):
        
        # Check if it is an Operator (Arity 2)
        if original_gene in available_operations:
            # "only can altered the function name ('add' -> subtract)"
            # Filter for operators distinct from the original
            candidates = [op for op in available_operations if op != original_gene]
            if candidates:
                new_gene = random.choice(candidates)
                
        # Otherwise, it must be a Heuristic Name (Arity 0)
        else:
            # "only the... dual bound name can be altered"
            # Replace with a different heuristic (e.g., 'h1' -> 'h3')
            dual_bound_names = list(dual_bound_functions_registry.keys())
            candidates = [h for h in dual_bound_names if h != original_gene]
            if candidates:
                new_gene = random.choice(candidates)
    
    # 5. Construct New Chromosome
    mutated_chromosome = before_mutation_chromosome.copy()
    mutated_chromosome[mutation_idx] = new_gene
    
    # 6. Return New Individual
    try:
        # Check if valid RPN
        new_individual = {'chromosome': mutated_chromosome, 'fitness': 0}
        compile_chromosome_to_useable_function(new_individual, dual_bound_functions_registry, print_code=False)
        return mutated_chromosome
    except Exception as e:
        print(f"Mutation produced invalid candidates: {e}")
        return before_mutated_chromosome

#### *Testing point mutation*

In [46]:
def h1(state): return sum(state)
def h2(state): return 20
def h3(state): return 5

dual_bound_functions_registry = {
    "h1": h1,
    "h2": h2,
    "h3": h3
}

# 1. Define Parents as per your example
# P1: [a1, h1, multiply, h2, add]
# Note: using strings for a1/h1 for visualization
parent1 = {
    'chromosome': [2, 'h1', 'MULTIPLY', 'h2', 'ADD'], # a1=2 for clarity
    'fitness': None
}

print(f"Parent 1: {parent1['chromosome']}")

# 2. Perform Crossover
child_chromosome = point_mutation(parent1, dual_bound_functions_registry, LB_range_of_constant, UB_range_of_constant)

Parent 1: [2, 'h1', 'MULTIPLY', 'h2', 'ADD']
Generated Code:
def dual_bound_combination(state):
    return ((9.4 * h1(state)) + h2(state))



## Combined mutation generator

In [47]:
def combined_mutation_generator(parent_dict, dual_bound_functions_registry, 
                                LB_range_of_constant, UB_range_of_constant,
                                didp_model_registry, dual_bound_expression_function,
                                mutation_max_subtree_depth = mutation_max_subtree_depth, 
                                reference_point = OPTIMAL_COST_REFERENCE, 
                                available_operations=available_operations):
    """
    Randomly selects one of the mutation methods to produce a mutated offspring,
    then evaluates the fitness of the resulting offspring.
    
    Returns:
        list: A list containing the single evaluated dictionary [{'chromosome': [...], 'fitness': float}]
    """
    mutation_methods = [
        "subtree",
        "point"
    ]
    
    # 1. Select Method
    selected_method = random.choice(mutation_methods)
    
    raw_mutated_chromosome = []
    
    # 2. Generate Mutated Offspring (Raw List or Dict)
    if selected_method == "subtree":
        # print("Using Subtree Mutation.")
        raw_mutated_chromosome = subtree_mutation(
            parent_dict, 
            dual_bound_functions_registry, 
            LB_range_of_constant, 
            UB_range_of_constant, 
        )
    
    elif selected_method == "point":
        # print("Using Point Mutation.")
        raw_mutated_chromosome = point_mutation(
            parent_dict, 
            dual_bound_functions_registry, 
            LB_range_of_constant, 
            UB_range_of_constant, 
        )

    # 3. Normalize Output (Handle cases where mutation returns the original dict vs a new list)
    if isinstance(raw_mutated_chromosome, dict):
        raw_mutated_offspring_chromosome = raw_mutated_chromosome.get('chromosome')
    else:
        raw_mutated_offspring_chromosome = raw_mutated_chromosome

    # 4. Evaluate Fitness
    # Create the dictionary structure
    raw_mutated_chromosome_fitness_dict = {'chromosome': raw_mutated_offspring_chromosome, 'fitness': None}
    
    # Calculate fitness
    evaluated_mutated_chromosome_fitness_dict = chromosome_fitness_dict_evaluation(
        raw_mutated_chromosome_fitness_dict, 
        didp_model_registry, 
        dual_bound_expression_function, 
        reference_point
    )
    
    # Return as a list to match the crossover generator's interface
    return [evaluated_mutated_chromosome_fitness_dict]

#### *Testing combined mutation generator*

In [48]:
Test_OPTIMAL_COST_REFERENCE = 5 
def creation_of_didp_model_function():
    import modified_didppy as m_dp
    # [Data definition]
    
    #[DIDP Model Creation]
    model = m_dp.Model()
    # Create an integer variable 'x'. Set the START STATE to 5.
    x = model.add_int_var(target=5)
    # Set the GOAL CONDITION to x == 0.
    model.add_base_case([x == 0])
    #
    # --- END OF FIX ---
    #
    model.add_transition(
        m_dp.Transition(
            name="decrement",
            cost=1 + m_dp.IntExpr.state_cost(), # Cost is 1 per step
            effects=[(x, x - 1)]
        )
    )
    # This line is not needed for forward search, so we remove it.
    # model.target_state[x] = 5 
    # 2. Add a standard Rust expression bound
    model.add_dual_bound(0) 
    # [Return variables needed for dual bounds calculation]
    didp_model = model 
    didp_model_metadata_dict = {"x": x}
    didp_bundle = (didp_model, didp_model_metadata_dict)
    return didp_bundle

def dual_bound_expression_function(didp_bundle):
    didp_model, didp_model_metadata_dict = didp_bundle
    # [Variables/data definition]
    x = didp_model_metadata_dict['x']
    #[Dual bound logic declaration]
    def h1(state):
        return state[x]+1 # Example logic
    def h2(state):
        return 20
    def h3(state):
        return 5
    # [Return a dual bound registry dict]
    dual_bound_dict = automatic_creation_of_dual_bounds_registry(locals())
    return dual_bound_dict
available_operations = ["ADD", "SUBTRACT", "MAX", "MIN", "MULTIPLY", "PDIV"]

dual_bound_functions_registry = dual_bound_expression_function(creation_of_didp_model_function())

parent1 = {
    'chromosome': [2, 'h1', 'MULTIPLY', 'h2', 'ADD'], # a1=2 for clarity
    'fitness': None
}

print(f"Parent 1: {parent1['chromosome']}")

print("--- Starting Combined Mutation Test ---")

# We run the generator. 
# It will:
# 1. Pick a crossover method (One Point, Subtree, or Uniform)
# 2. Generate offspring
# 3. Create a NEW didp model using 'creation_of_didp_model_function'
# 4. Link heuristics using 'dual_bound_expression_function'
# 5. Solve and calculate fitness
evaluated_offspring = combined_mutation_generator(parent1, dual_bound_functions_registry, 
                                LB_range_of_constant, UB_range_of_constant,
                                didp_model_registry = creation_of_didp_model_function,   # Pass the model factory
                                dual_bound_expression_function = dual_bound_expression_function,   # Pass the heuristic factory
                                reference_point = Test_OPTIMAL_COST_REFERENCE
)

# --- 5. Results ---
print(f"\nGenerated {len(evaluated_offspring)} offspring.")
for i, child in enumerate(evaluated_offspring):
    print(f"\nChild {i+1}:")
    print(f"  Chromosome: {child['chromosome']}")
    print(f"  Fitness:    {child['fitness']}")
    
    # Interpretation
    if child['fitness'] is not None and child['fitness'] < 1e-5:
        print("  -> Status: Optimal Solution Found (Fitness ~ 0)")
    elif child['fitness'] == float('inf'):
        print("  -> Status: Invalid/Infeasible")
    else:
        print(f"  -> Status: Valid but Suboptimal (Deviation: {child['fitness']})")

Parent 1: [2, 'h1', 'MULTIPLY', 'h2', 'ADD']
--- Starting Combined Mutation Test ---
Generated Code:
def dual_bound_combination(state):
    return ((2 * h1(state)) + h1(state))


Generated 1 offspring.

Child 1:
  Chromosome: [2, 'h1', 'MULTIPLY', 'h1', 'ADD']
  Fitness:    0.0
  -> Status: Optimal Solution Found (Fitness ~ 0)


# **Evolutionary algorithm execution**

In [ ]:
def evolution_algorithm_execution(
    # --- 1. Algorithm Hyperparameters ---
    population_size=POPULATION_SIZE,
    generations=GENERATIONS,
    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
    elitism_rate=ELITISM_RATE, # 1% of best individuals preserved
    
    # --- 2. Generation Constraints ---
    LB_range_of_constant=LB_range_of_constant,
    UB_range_of_constant=UB_range_of_constant,
    min_chromosome_length=min_chromosome_length,
    max_chromosome_length=max_chromosome_length,
    mutation_max_subtree_depth=mutation_max_subtree_depth,
    available_operations=available_operations,
    
    # --- 3. Problem Domain (Factories & Logic) ---
    didp_model_registry=None,          # MUST BE PROVIDED: Factory function creation_of_didp_model_function
    dual_bound_expression_function=None, # MUST BE PROVIDED: Factory function dual_bound_expression_function
    reference_point=OPTIMAL_COST_REFERENCE,
    solver_time_limit=SOLVER_TIME_LIMIT,
    
    # --- 4. Operator Probabilities ---
    homology_1_point_crossover_probability=homology_1_point_crossover_probability,
    subtree_crossover_probability=subtree_crossover_probability,
    uniform_crossover_probability=uniform_crossover_probability,
    ):
    """
    Executes the full Evolutionary Algorithm lifecycle for Domain-Independent Dynamic Programming.
    """
    
    # --- STEP 0: PREPARE REGISTRY ---
    # We create a static registry ONCE to validate syntax during generation/crossover/mutation.
    # The actual fitness evaluation will create its own fresh registry per model instance.
    if didp_model_registry is None or dual_bound_expression_function is None:
        raise ValueError("You must provide 'didp_model_registry' and 'dual_bound_expression_function' factories.")
        
    static_model_bundle = didp_model_registry()
    dual_bound_functions_registry = dual_bound_expression_function(static_model_bundle)
    
    print(f"--- Initialization: Generating Population of size {population_size} ---")
    
    # --- STEP 1: INITIALIZATION ---
    population = initialize_list_of_chromosome_fitness_dictionary(
        list_size = population_size, 
        dual_bound_functions_dict = dual_bound_functions_registry,
        LB_range_of_constant=LB_range_of_constant, 
        UB_range_of_constant=UB_range_of_constant,
        didp_model_registry = didp_model_registry, 
        dual_bound_expression_function = dual_bound_expression_function,
        min_chromosome_length=min_chromosome_length, 
        max_chromosome_length = max_chromosome_length, 
        available_operations = available_operations,
        reference_point = OPTIMAL_COST_REFERENCE
    )
    
    # Track global best
    best_individual_ever = min(population, key=lambda x: x['fitness'] if x['fitness'] is not None else float('inf'))
    print(f"Initial Best Fitness: {best_individual_ever['fitness']}")

    # --- STEP 2: GENERATIONAL LOOP ---
    for gen in range(1, generations + 1):
        
        new_population = []
        
        # --- A. REPRODUCTION (Elitism) ---
        # Keep top 1% (or at least 1) best parents directly
        num_elites = max(1, int(population_size * elitism_rate))
        # Sort by fitness (lower is better)
        sorted_pop = sorted(population, key=lambda x: x['fitness'] if x['fitness'] is not None else float('inf'))
        elites = sorted_pop[:num_elites]
        new_population.extend(elites)
        
        # --- B. OFFSPRING GENERATION ---
        # We need to fill the rest of the population
        while len(new_population) < population_size:
            
            # 1. Selection
            parent1 = parents_selection(population)
            parent2 = parents_selection(population)
            
            # 2. Crossover
            if random.random() < crossover_rate:
                # Returns a LIST of evaluated offspring
                offspring_list = combined_crossover_generator(
                    parent1, 
                    parent2, 
                    dual_bound_functions_registry = dual_bound_functions_registry,
                    didp_model_registry = didp_model_registry, 
                    dual_bound_expression_function = dual_bound_expression_function, 
                    reference_point = OPTIMAL_COST_REFERENCE,
                    homology_1_point_crossover_probability = homology_1_point_crossover_probability,
                    subtree_crossover_probability = subtree_crossover_probability,
                    uniform_crossover_probability = uniform_crossover_probability,
                    available_operations = available_operations
                )
            else:
                # No Crossover: Just clone parents
                # We re-evaluate them just in case (or you could copy fitness)
                offspring_list = [parent1, parent2] 
            
            # 3. Mutation
            final_offspring_for_batch = []
            for ind in offspring_list:
                if random.random() < mutation_rate:
                    # Returns a LIST containing the single mutated offspring (evaluated)
                    mutated_list = combined_mutation_generator(
                        parent_dict=ind, 
                        dual_bound_functions_registry=dual_bound_functions_registry,
                        LB_range_of_constant = LB_range_of_constant,
                        UB_range_of_constant = UB_range_of_constant,
                        didp_model_registry = didp_model_registry,
                        dual_bound_expression_function = dual_bound_expression_function,
                        mutation_max_subtree_depth = mutation_max_subtree_depth,
                        reference_point = OPTIMAL_COST_REFERENCE,
                        available_operations= available_operations
                    )
                    final_offspring_for_batch.append(mutated_list[0])
                else:
                    final_offspring_for_batch.append(ind)
            
            # 4. Add to New Population
            for child in final_offspring_for_batch:
                if len(new_population) < population_size:
                    new_population.append(child)
        
        # --- STEP 3: UPDATE & LOGGING ---
        population = new_population
        
        # Find best in current generation
        current_best = min(population, key=lambda x: x['fitness'] if x['fitness'] is not None else float('inf'))
        
        # Update global best
        if (current_best['fitness'] is not None and 
            (best_individual_ever['fitness'] is None or current_best['fitness'] < best_individual_ever['fitness'])):
            best_individual_ever = current_best
            
        print(f"Gen {gen}: Best Fitness = {current_best['fitness']} | Global Best = {best_individual_ever['fitness']}")
        
        # Optional: Early Stopping if optimal found
        if best_individual_ever['fitness'] is not None and best_individual_ever['fitness'] < 1e-6:
            print("Optimal solution found! Stopping early.")
            break

    print("--- Evolution Completed ---")
    print(f"Best Individual Found: {best_individual_ever}")
    return best_individual_ever

#### *Testing Evolutionary algorithm*

In [55]:
# ==========================================
# 1. EVOLUTIONARY ALGORITHM HYPERPARAMETERS
# ==========================================
POPULATION_SIZE = 20        # Size of the population in each generation
GENERATIONS = 2          # Number of generations to run
MUTATION_RATE = 0.2         # Probability of mutating an individual
CROSSOVER_RATE = 0.8        # Probability of performing crossover
ELITISM_RATE = 0.01
# ==========================================
# 2. CHROMOSOME GENERATION CONSTRAINTS
# ==========================================
# Bounds for the coefficients generated for weighted blocks (e.g., 5.5 * h1)
LB_range_of_constant = 0.0  
UB_range_of_constant = 10.0 

# Depth limits for the RPN trees (used in Ramped Half-and-Half generator)
min_chromosome_length = 2               # Minimum depth of the initial trees
max_chromosome_length = 6               # Maximum depth of the initial trees
# Mutation: Maximum depth allowed for the *newly generated* subtree during mutation
mutation_max_subtree_depth = random.randint(min_chromosome_length, max_chromosome_length + 4)  # Randomly chosen between 1 and 3

# ==========================================
# 3. PROBLEM DOMAIN (INPUTS)
# ==========================================
# The atomic operations allowed in the RPN expression
available_operations = ["ADD", "SUBTRACT", "MAX", "MIN", "MULTIPLY", "PDIV"]

# The Ground Truth optimal cost for the specific problem instance
# Used to calculate fitness (deviation from optimal)
OPTIMAL_COST_REFERENCE = 5 

# Time limit (in seconds) for the DIDP solver to run per chromosome evaluation
SOLVER_TIME_LIMIT = 0.5 #seconds

# ==========================================
# 4. OPERATOR SPECIFIC PARAMETERS
# ==========================================
# 1-Point Crossover: Probability of using Homology (matching structure) vs Random fallback
homology_1_point_crossover_probability = 0.5

# Subtree Crossover: Probability of swapping a Function (Branch) vs Terminal (Leaf)
subtree_crossover_probability = 0.9

# Uniform Crossover: Probability of swapping genes at a specific index
uniform_crossover_probability = 0.5

def creation_of_didp_model_function():
    import modified_didppy as m_dp
    # [Data definition]
    
    #[DIDP Model Creation]
    model = m_dp.Model()
    # Create an integer variable 'x'. Set the START STATE to 5.
    x = model.add_int_var(target=5)
    # Set the GOAL CONDITION to x == 0.
    model.add_base_case([x == 0])
    #
    # --- END OF FIX ---
    #
    model.add_transition(
        m_dp.Transition(
            name="decrement",
            cost=1 + m_dp.IntExpr.state_cost(), # Cost is 1 per step
            effects=[(x, x - 1)]
        )
    )
    # This line is not needed for forward search, so we remove it.
    # model.target_state[x] = 5 
    # 2. Add a standard Rust expression bound
    model.add_dual_bound(0) 
    # [Return variables needed for dual bounds calculation]
    didp_model = model 
    didp_model_metadata_dict = {"x": x}
    didp_bundle = (didp_model, didp_model_metadata_dict)
    return didp_bundle

def dual_bound_expression_function(didp_bundle):
    didp_model, didp_model_metadata_dict = didp_bundle
    # [Variables/data definition]
    x = didp_model_metadata_dict['x']
    #[Dual bound logic declaration]
    def h1(state):
        return state[x]+1 # Example logic
    def h2(state):
        return 20
    def h3(state):
        return 5
    # [Return a dual bound registry dict]
    dual_bound_dict = automatic_creation_of_dual_bounds_registry(locals())
    return dual_bound_dict
available_operations = ["ADD", "SUBTRACT", "MAX", "MIN", "MULTIPLY", "PDIV"]

dual_bound_functions_registry = dual_bound_expression_function(creation_of_didp_model_function())

best_individual_ever = evolution_algorithm_execution(
    # --- 1. Algorithm Hyperparameters ---
    population_size=POPULATION_SIZE,
    generations=GENERATIONS,
    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
    elitism_rate=ELITISM_RATE, # 1% of best individuals preserved
    
    # --- 2. Generation Constraints ---
    LB_range_of_constant=LB_range_of_constant,
    UB_range_of_constant=UB_range_of_constant,
    min_chromosome_length=min_chromosome_length,
    max_chromosome_length=max_chromosome_length,
    mutation_max_subtree_depth=mutation_max_subtree_depth,
    available_operations=available_operations,
    
    # --- 3. Problem Domain (Factories & Logic) ---
    didp_model_registry=creation_of_didp_model_function,          # MUST BE PROVIDED: Factory function creation_of_didp_model_function
    dual_bound_expression_function=dual_bound_expression_function, # MUST BE PROVIDED: Factory function dual_bound_expression_function
    reference_point=OPTIMAL_COST_REFERENCE,
    solver_time_limit=SOLVER_TIME_LIMIT,
    
    # --- 4. Operator Probabilities ---
    homology_1_point_crossover_probability=homology_1_point_crossover_probability,
    subtree_crossover_probability=subtree_crossover_probability,
    uniform_crossover_probability=uniform_crossover_probability
)

--- Initialization: Generating Population of size 20 ---
Initial Best Fitness: 0.0
Generated Code:
def dual_bound_combination(state):
    return (h2(state) + min(((min(max(h2(state), h2(state)), ((8.73 * h3(state)) + h2(state))) / (((4.86 * h2(state)) * h3(state)) + ((2.16 * h1(state)) + h3(state))) if abs((((4.86 * h2(state)) * h3(state)) + ((2.16 * h1(state)) + h3(state)))) > 1e-6 else 1.0) + (min((h2(state) * (4.07 * h2(state))), (h1(state) - (0.13 * h2(state)))) / min(min((9.29 * h1(state)), (4.85 * h2(state))), ((7.03 * h1(state)) - h2(state))) if abs(min(min((9.29 * h1(state)), (4.85 * h2(state))), ((7.03 * h1(state)) - h2(state)))) > 1e-6 else 1.0)), max((min(((5.72 * h2(state)) / h1(state) if abs(h1(state)) > 1e-6 else 1.0), min(h1(state), h2(state))) / min((h2(state) + h3(state)), (h2(state) / (9.54 * h1(state)) if abs((9.54 * h1(state))) > 1e-6 else 1.0)) if abs(min((h2(state) + h3(state)), (h2(state) / (9.54 * h1(state)) if abs((9.54 * h1(state))) > 1e-6 else 1.0))) > 1e-6 e